# AutoDS - Completed Work Notebook

**Dataset:** Unclean Dataset 1.xlsx  
**Generated:** 2026-10-20 10:08  
**Rows:** 126  **Columns:** 7

This notebook keeps the export focused on the Python code and results from this session.

**Setup:** unzip this alongside the CSV file(s) it downloaded with (same folder) and run the cells from top to bottom -- no other setup needed. The zip also includes an `autods_cleaning_tools/` folder (the app's own cleaning code) that the cleaning cells below import from -- keep it next to the notebook too.

In [ ]:
import pandas as pd

# The original data exactly as uploaded, before any AutoDS cleaning.
# Place this notebook in the same folder as the CSV file(s) it downloaded with.
df_original = pd.read_csv('original_data.csv')
df = df_original.copy()

print(f'Loaded data: {len(df):,} rows x {len(df.columns)} columns')
df.head()

## Notes

**Sum of total_payment by gender, across course**

| gender | Cyber Security | Data Analysis | Data Science | Machine Learning | Web Development | Total |
| --- | --- | --- | --- | --- | --- | --- |
| F | 17726.51 | 16125.92 | 11864.98 | 40292.59 | 2910.14 | 88920.14 |
| M | 35875.03 | 42587.79 | 29449.49 | 5514.61 | 41622.15 | 155049.07 |
| Total | 53601.54 | 58713.71 | 41314.47 | 45807.2 | 44532.29 | 243969.21 |

Data Analysis is the heighest revenue generating course while Web Development is the list

## Cleaning

This reruns the exact cleaning steps applied in AutoDS, using the app's own `clean_dataframe()` (the "Clean Now" button) and `run_command()` (the chat cleaning assistant) -- the same functions AutoDS itself calls, bundled into `autods_cleaning_tools/` next to this notebook -- so the result matches the app instead of approximating it.

_Note: AutoDS doesn't record whether "Clean Now" was clicked before or after chat commands, only that both happened -- they're replayed here as Clean Now first, then every chat command in the order it was run._

In [ ]:
import sys
sys.path.insert(0, 'autods_cleaning_tools')
from clean import clean_dataframe
from clean_chat import run_command


### Clean Now (automated cleaning)

In [ ]:
clean_options = {
    'fix_column_names': True,
    'strip_whitespace': True,
    'fix_mixed_types': True,
    'drop_constant_cols': True,
    'drop_high_missing_cols': True,
    'drop_duplicates': True,
    'fill_missing_numeric': 'median',
    'fill_missing_categorical': 'mode',
    'remove_outliers': True,
}

df, log = clean_dataframe(df, clean_options)
print('\n'.join(log))


### Chat cleaning commands

In [ ]:
df, message = run_command(df, 'replace 4 with Data Science in course', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'fill null in Enrollment_Date with 2022-06-25', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'remove row 127', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'fill null in age with median', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'replace = with unknow in first_name', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'merge first_name and last name into Full_Name', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'drop column first name', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'drop column last name', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'replace Web Developmen with Web Development in course', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'replace Web Develpment with Web Developmentin course', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'replace Web Develpment with Web Development in course', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'replace Data Analytics with Data Analysis in course', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'replace f with F in gender', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'replace Web Developmet with Web Development', original_df=df_original)
print(message)


In [ ]:
df, message = run_command(df, 'replace Web Developmet with Web Development in course', original_df=df_original)
print(message)


In [ ]:
print(f'Final shape after replaying AutoDS cleaning: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()


## Visualizations Done (5)

In [ ]:
import json
import matplotlib.pyplot as plt

chart = json.loads("{\"title\": \"Distribution of gender\", \"reason\": \"\", \"type\": \"pie\", \"x\": \"gender\", \"caption\": \"Share of each category in 'gender' \u2014 slice size = proportion of rows.\", \"labels\": [\"M\", \"F\"], \"values\": [74, 52], \"_reportSource\": \"auto\", \"user_note\": \"The pie charts shows distribution of gender. Male(M) are the majority of the students with 74 out of 126 students being male. The number of females are 52. This  shows the difference between the two gender is 22\"}")
chart_type = chart.get('type')
x = chart.get('x')
y = chart.get('y')
_plotted = False  # set True below once a chart type renders via a non-matplotlib path

if chart_type in {'scatter', 'line'} and x in df.columns and y in df.columns:
    plot_df = df[[x, y]].dropna()
    if chart_type == 'line':
        plot_df = plot_df.sort_values(x)
        plt.plot(plot_df[x], plot_df[y], marker='o', color=chart.get('color') or None)
    else:
        plt.scatter(plot_df[x], plot_df[y], alpha=0.7, color=chart.get('color') or None)
    plt.xlabel(x)
    plt.ylabel(y)
elif chart_type == 'histogram' and x in df.columns:
    df[x].dropna().hist(bins=12, color=chart.get('color') or None)
    plt.xlabel(x)
elif chart_type == 'boxplot' and x in df.columns:
    # Replay using the SAME mode and grouping the person actually picked
    # (stored on the chart itself) -- 'simple' mode plots the sum per
    # group (matching a pivot table, with no quartiles/median/outliers),
    # 'full' mode replays an actual statistical box-and-whisker plot,
    # grouped the same way it was grouped live.
    _group_col = chart.get('group')
    _is_simple = chart.get('_boxplotMode') == 'simple'
    if _group_col and _group_col in df.columns:
        if _is_simple:
            _sums = df.groupby(_group_col)[x].sum().sort_values(ascending=False)
            plt.bar(_sums.index.astype(str), _sums.values, color=chart.get('color') or None)
            plt.ylabel(f'Sum of {x}')
            plt.xlabel(_group_col)
        else:
            df.boxplot(column=x, by=_group_col)
            plt.title('')
            plt.suptitle('')
            plt.ylabel(x)
            plt.xlabel(_group_col)
    elif _is_simple:
        _total = df[x].dropna().sum()
        plt.bar([x], [_total], color=chart.get('color') or None)
        plt.ylabel(f'Sum of {x}')
    else:
        df[x].dropna().plot(kind='box')
        plt.xlabel(x)
elif chart_type in {'bar', 'word_frequency'} and chart.get('labels'):
    labels = chart.get('labels', [])
    values = chart.get('values', [])
    _cs = chart.get('colors')
    plt.barh(labels[::-1], values[::-1], color=(_cs[::-1] if _cs else chart.get('color')) or None)
    plt.xlabel((chart.get('y_label') or chart.get('y')) if chart_type == 'bar' else 'Count')
elif chart_type == 'stacked_bar' and chart.get('series'):
    import pandas as _pd
    _labels = chart.get('labels', [])
    _stacked_df = _pd.DataFrame({s['label']: s['values'] for s in chart['series']}, index=_labels)
    _stacked_df.plot(kind='bar', stacked=True, ax=plt.gca())
    plt.xlabel(x)
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title=chart.get('group'), bbox_to_anchor=(1.02, 1), loc='upper left')
elif chart_type == 'wordcloud' and chart.get('words'):
    words = chart.get('words', [])
    labels = [w.get('word') for w in words][:25][::-1]
    values = [w.get('count') for w in words][:25][::-1]
    plt.barh(labels, values, color=chart.get('colors') or chart.get('color') or None)
    plt.xlabel('Count')
elif chart_type == 'choropleth' and chart.get('rows'):
    # Renders as an actual interactive map via Plotly Express, which ships
    # its own world/US-state boundaries -- no geopandas or shapefile needed.
    # Falls back to a static ranked bar chart if plotly isn't installed, or
    # if this map used a custom-uploaded .geojson whose shapes Plotly has no
    # built-in knowledge of.
    value_col = chart.get('value_col')
    name_col = chart.get('name_col')
    level = chart.get('level')
    rows = [r for r in chart['rows'] if r.get(value_col) is not None and r.get(name_col)]
    _NAME_TO_ISO3 = json.loads("{\"Zimbabwe\": \"ZWE\", \"Zambia\": \"ZMB\", \"Yemen\": \"YEM\", \"Vietnam\": \"VNM\", \"Venezuela\": \"VEN\", \"Vatican\": \"VAT\", \"Vanuatu\": \"VUT\", \"Uzbekistan\": \"UZB\", \"Uruguay\": \"URY\", \"Federated States of Micronesia\": \"FSM\", \"Marshall Islands\": \"MHL\", \"Northern Mariana Islands\": \"MNP\", \"United States Virgin Islands\": \"VIR\", \"Guam\": \"GUM\", \"American Samoa\": \"ASM\", \"Puerto Rico\": \"PRI\", \"United States of America\": \"USA\", \"South Georgia and the Islands\": \"SGS\", \"British Indian Ocean Territory\": \"IOT\", \"Saint Helena\": \"SHN\", \"Pitcairn Islands\": \"PCN\", \"Anguilla\": \"AIA\", \"Falkland Islands\": \"FLK\", \"Cayman Islands\": \"CYM\", \"Bermuda\": \"BMU\", \"British Virgin Islands\": \"VGB\", \"Turks and Caicos Islands\": \"TCA\", \"Montserrat\": \"MSR\", \"Jersey\": \"JEY\", \"Guernsey\": \"GGY\", \"Isle of Man\": \"IMN\", \"United Kingdom\": \"GBR\", \"United Arab Emirates\": \"ARE\", \"Ukraine\": \"UKR\", \"Uganda\": \"UGA\", \"Turkmenistan\": \"TKM\", \"Turkey\": \"TUR\", \"Tunisia\": \"TUN\", \"Trinidad and Tobago\": \"TTO\", \"Tonga\": \"TON\", \"Togo\": \"TGO\", \"East Timor\": \"TLS\", \"Thailand\": \"THA\", \"United Republic of Tanzania\": \"TZA\", \"Tajikistan\": \"TJK\", \"Taiwan\": \"TWN\", \"Syria\": \"SYR\", \"Switzerland\": \"CHE\", \"Sweden\": \"SWE\", \"Swaziland\": \"SWZ\", \"Suriname\": \"SUR\", \"South Sudan\": \"SSD\", \"Sudan\": \"SDN\", \"Sri Lanka\": \"LKA\", \"Spain\": \"ESP\", \"South Korea\": \"KOR\", \"South Africa\": \"ZAF\", \"Somalia\": \"SOM\", \"Somaliland\": \"SOL\", \"Solomon Islands\": \"SLB\", \"Slovakia\": \"SVK\", \"Slovenia\": \"SVN\", \"Singapore\": \"SGP\", \"Sierra Leone\": \"SLE\", \"Seychelles\": \"SYC\", \"Republic of Serbia\": \"SRB\", \"Senegal\": \"SEN\", \"Saudi Arabia\": \"SAU\", \"S\u00e3o Tom\u00e9 and Principe\": \"STP\", \"San Marino\": \"SMR\", \"Samoa\": \"WSM\", \"Saint Vincent and the Grenadines\": \"VCT\", \"Saint Lucia\": \"LCA\", \"Saint Kitts and Nevis\": \"KNA\", \"Rwanda\": \"RWA\", \"Russia\": \"RUS\", \"Romania\": \"ROU\", \"Qatar\": \"QAT\", \"Portugal\": \"PRT\", \"Poland\": \"POL\", \"Philippines\": \"PHL\", \"Peru\": \"PER\", \"Paraguay\": \"PRY\", \"Papua New Guinea\": \"PNG\", \"Panama\": \"PAN\", \"Palau\": \"PLW\", \"Pakistan\": \"PAK\", \"Oman\": \"OMN\", \"Norway\": \"NOR\", \"North Korea\": \"PRK\", \"Nigeria\": \"NGA\", \"Niger\": \"NER\", \"Nicaragua\": \"NIC\", \"New Zealand\": \"NZL\", \"Niue\": \"NIU\", \"Cook Islands\": \"COK\", \"Netherlands\": \"NLD\", \"Aruba\": \"ABW\", \"Cura\u00e7ao\": \"CUW\", \"Nepal\": \"NPL\", \"Nauru\": \"NRU\", \"Namibia\": \"NAM\", \"Mozambique\": \"MOZ\", \"Morocco\": \"MAR\", \"Western Sahara\": \"ESH\", \"Montenegro\": \"MNE\", \"Mongolia\": \"MNG\", \"Moldova\": \"MDA\", \"Monaco\": \"MCO\", \"Mexico\": \"MEX\", \"Mauritius\": \"MUS\", \"Mauritania\": \"MRT\", \"Malta\": \"MLT\", \"Mali\": \"MLI\", \"Maldives\": \"MDV\", \"Malaysia\": \"MYS\", \"Malawi\": \"MWI\", \"Madagascar\": \"MDG\", \"Macedonia\": \"MKD\", \"Luxembourg\": \"LUX\", \"Lithuania\": \"LTU\", \"Liechtenstein\": \"LIE\", \"Libya\": \"LBY\", \"Liberia\": \"LBR\", \"Lesotho\": \"LSO\", \"Lebanon\": \"LBN\", \"Latvia\": \"LVA\", \"Laos\": \"LAO\", \"Kyrgyzstan\": \"KGZ\", \"Kuwait\": \"KWT\", \"Kosovo\": \"KOS\", \"Kiribati\": \"KIR\", \"Kenya\": \"KEN\", \"Kazakhstan\": \"KAZ\", \"Jordan\": \"JOR\", \"Japan\": \"JPN\", \"Jamaica\": \"JAM\", \"Italy\": \"ITA\", \"Israel\": \"ISR\", \"Palestine\": \"PSE\", \"Ireland\": \"IRL\", \"Iraq\": \"IRQ\", \"Iran\": \"IRN\", \"Indonesia\": \"IDN\", \"India\": \"IND\", \"Iceland\": \"ISL\", \"Hungary\": \"HUN\", \"Honduras\": \"HND\", \"Haiti\": \"HTI\", \"Guyana\": \"GUY\", \"Guinea-Bissau\": \"GNB\", \"Guinea\": \"GIN\", \"Guatemala\": \"GTM\", \"Grenada\": \"GRD\", \"Greece\": \"GRC\", \"Ghana\": \"GHA\", \"Germany\": \"DEU\", \"Georgia\": \"GEO\", \"Gambia\": \"GMB\", \"Gabon\": \"GAB\", \"France\": \"FRA\", \"Saint Pierre and Miquelon\": \"SPM\", \"Wallis and Futuna\": \"WLF\", \"Saint Martin\": \"MAF\", \"Saint Barthelemy\": \"BLM\", \"French Polynesia\": \"PYF\", \"New Caledonia\": \"NCL\", \"French Southern and Antarctic Lands\": \"ATF\", \"Aland\": \"ALA\", \"Finland\": \"FIN\", \"Fiji\": \"FJI\", \"Ethiopia\": \"ETH\", \"Estonia\": \"EST\", \"Eritrea\": \"ERI\", \"Equatorial Guinea\": \"GNQ\", \"El Salvador\": \"SLV\", \"Egypt\": \"EGY\", \"Ecuador\": \"ECU\", \"Dominican Republic\": \"DOM\", \"Dominica\": \"DMA\", \"Djibouti\": \"DJI\", \"Greenland\": \"GRL\", \"Faroe Islands\": \"FRO\", \"Denmark\": \"DNK\", \"Czech Republic\": \"CZE\", \"Northern Cyprus\": \"CYN\", \"Cyprus\": \"CYP\", \"Cuba\": \"CUB\", \"Croatia\": \"HRV\", \"Ivory Coast\": \"CIV\", \"Costa Rica\": \"CRI\", \"Democratic Republic of the Congo\": \"COD\", \"Republic of the Congo\": \"COG\", \"Comoros\": \"COM\", \"Colombia\": \"COL\", \"China\": \"CHN\", \"Macao S.A.R\": \"MAC\", \"Hong Kong S.A.R.\": \"HKG\", \"Chile\": \"CHL\", \"Chad\": \"TCD\", \"Central African Republic\": \"CAF\", \"Cape Verde\": \"CPV\", \"Canada\": \"CAN\", \"Cameroon\": \"CMR\", \"Cambodia\": \"KHM\", \"Myanmar\": \"MMR\", \"Burundi\": \"BDI\", \"Burkina Faso\": \"BFA\", \"Bulgaria\": \"BGR\", \"Brunei\": \"BRN\", \"Brazil\": \"BRA\", \"Botswana\": \"BWA\", \"Bosnia and Herzegovina\": \"BIH\", \"Bolivia\": \"BOL\", \"Bhutan\": \"BTN\", \"Benin\": \"BEN\", \"Belize\": \"BLZ\", \"Belgium\": \"BEL\", \"Belarus\": \"BLR\", \"Barbados\": \"BRB\", \"Bangladesh\": \"BGD\", \"Bahrain\": \"BHR\", \"The Bahamas\": \"BHS\", \"Azerbaijan\": \"AZE\", \"Austria\": \"AUT\", \"Australia\": \"AUS\", \"Indian Ocean Territories\": \"AUS\", \"Heard Island and McDonald Islands\": \"HMD\", \"Norfolk Island\": \"NFK\", \"Ashmore and Cartier Islands\": \"AUS\", \"Armenia\": \"ARM\", \"Argentina\": \"ARG\", \"Antigua and Barbuda\": \"ATG\", \"Angola\": \"AGO\", \"Andorra\": \"AND\", \"Algeria\": \"DZA\", \"Albania\": \"ALB\", \"Afghanistan\": \"AFG\", \"Siachen Glacier\": \"KAS\", \"Antarctica\": \"ATA\", \"Sint Maarten\": \"SXM\", \"Tuvalu\": \"TUV\"}")
    _ISO3_TO_NAME = json.loads("{\"ZWE\": \"zimbabwe\", \"ZMB\": \"zambia\", \"YEM\": \"yemen\", \"VNM\": \"vietnam\", \"VEN\": \"venezuela\", \"VAT\": \"vatican\", \"VUT\": \"vanuatu\", \"UZB\": \"uzbekistan\", \"URY\": \"uruguay\", \"FSM\": \"federated states of micronesia\", \"MHL\": \"marshall islands\", \"MNP\": \"northern mariana islands\", \"VIR\": \"united states virgin islands\", \"GUM\": \"guam\", \"ASM\": \"american samoa\", \"PRI\": \"puerto rico\", \"USA\": \"united states of america\", \"SGS\": \"south georgia and the islands\", \"IOT\": \"british indian ocean territory\", \"SHN\": \"saint helena\", \"PCN\": \"pitcairn islands\", \"AIA\": \"anguilla\", \"FLK\": \"falkland islands\", \"CYM\": \"cayman islands\", \"BMU\": \"bermuda\", \"VGB\": \"british virgin islands\", \"TCA\": \"turks and caicos islands\", \"MSR\": \"montserrat\", \"JEY\": \"jersey\", \"GGY\": \"guernsey\", \"IMN\": \"isle of man\", \"GBR\": \"united kingdom\", \"ARE\": \"united arab emirates\", \"UKR\": \"ukraine\", \"UGA\": \"uganda\", \"TKM\": \"turkmenistan\", \"TUR\": \"turkey\", \"TUN\": \"tunisia\", \"TTO\": \"trinidad and tobago\", \"TON\": \"tonga\", \"TGO\": \"togo\", \"TLS\": \"east timor\", \"THA\": \"thailand\", \"TZA\": \"united republic of tanzania\", \"TJK\": \"tajikistan\", \"TWN\": \"taiwan\", \"SYR\": \"syria\", \"CHE\": \"switzerland\", \"SWE\": \"sweden\", \"SWZ\": \"swaziland\", \"SUR\": \"suriname\", \"SSD\": \"south sudan\", \"SDN\": \"sudan\", \"LKA\": \"sri lanka\", \"ESP\": \"spain\", \"KOR\": \"south korea\", \"ZAF\": \"south africa\", \"SOM\": \"somalia\", \"SOL\": \"somaliland\", \"SLB\": \"solomon islands\", \"SVK\": \"slovakia\", \"SVN\": \"slovenia\", \"SGP\": \"singapore\", \"SLE\": \"sierra leone\", \"SYC\": \"seychelles\", \"SRB\": \"republic of serbia\", \"SEN\": \"senegal\", \"SAU\": \"saudi arabia\", \"STP\": \"s\u00e3o tom\u00e9 and principe\", \"SMR\": \"san marino\", \"WSM\": \"samoa\", \"VCT\": \"saint vincent and the grenadines\", \"LCA\": \"saint lucia\", \"KNA\": \"saint kitts and nevis\", \"RWA\": \"rwanda\", \"RUS\": \"russia\", \"ROU\": \"romania\", \"QAT\": \"qatar\", \"PRT\": \"portugal\", \"POL\": \"poland\", \"PHL\": \"philippines\", \"PER\": \"peru\", \"PRY\": \"paraguay\", \"PNG\": \"papua new guinea\", \"PAN\": \"panama\", \"PLW\": \"palau\", \"PAK\": \"pakistan\", \"OMN\": \"oman\", \"NOR\": \"norway\", \"PRK\": \"north korea\", \"NGA\": \"nigeria\", \"NER\": \"niger\", \"NIC\": \"nicaragua\", \"NZL\": \"new zealand\", \"NIU\": \"niue\", \"COK\": \"cook islands\", \"NLD\": \"netherlands\", \"ABW\": \"aruba\", \"CUW\": \"cura\u00e7ao\", \"NPL\": \"nepal\", \"NRU\": \"nauru\", \"NAM\": \"namibia\", \"MOZ\": \"mozambique\", \"MAR\": \"morocco\", \"ESH\": \"western sahara\", \"MNE\": \"montenegro\", \"MNG\": \"mongolia\", \"MDA\": \"moldova\", \"MCO\": \"monaco\", \"MEX\": \"mexico\", \"MUS\": \"mauritius\", \"MRT\": \"mauritania\", \"MLT\": \"malta\", \"MLI\": \"mali\", \"MDV\": \"maldives\", \"MYS\": \"malaysia\", \"MWI\": \"malawi\", \"MDG\": \"madagascar\", \"MKD\": \"macedonia\", \"LUX\": \"luxembourg\", \"LTU\": \"lithuania\", \"LIE\": \"liechtenstein\", \"LBY\": \"libya\", \"LBR\": \"liberia\", \"LSO\": \"lesotho\", \"LBN\": \"lebanon\", \"LVA\": \"latvia\", \"LAO\": \"laos\", \"KGZ\": \"kyrgyzstan\", \"KWT\": \"kuwait\", \"KOS\": \"kosovo\", \"KIR\": \"kiribati\", \"KEN\": \"kenya\", \"KAZ\": \"kazakhstan\", \"JOR\": \"jordan\", \"JPN\": \"japan\", \"JAM\": \"jamaica\", \"ITA\": \"italy\", \"ISR\": \"israel\", \"PSE\": \"palestine\", \"IRL\": \"ireland\", \"IRQ\": \"iraq\", \"IRN\": \"iran\", \"IDN\": \"indonesia\", \"IND\": \"india\", \"ISL\": \"iceland\", \"HUN\": \"hungary\", \"HND\": \"honduras\", \"HTI\": \"haiti\", \"GUY\": \"guyana\", \"GNB\": \"guinea-bissau\", \"GIN\": \"guinea\", \"GTM\": \"guatemala\", \"GRD\": \"grenada\", \"GRC\": \"greece\", \"GHA\": \"ghana\", \"DEU\": \"germany\", \"GEO\": \"georgia\", \"GMB\": \"gambia\", \"GAB\": \"gabon\", \"FRA\": \"france\", \"SPM\": \"saint pierre and miquelon\", \"WLF\": \"wallis and futuna\", \"MAF\": \"saint martin\", \"BLM\": \"saint barthelemy\", \"PYF\": \"french polynesia\", \"NCL\": \"new caledonia\", \"ATF\": \"french southern and antarctic lands\", \"ALA\": \"aland\", \"FIN\": \"finland\", \"FJI\": \"fiji\", \"ETH\": \"ethiopia\", \"EST\": \"estonia\", \"ERI\": \"eritrea\", \"GNQ\": \"equatorial guinea\", \"SLV\": \"el salvador\", \"EGY\": \"egypt\", \"ECU\": \"ecuador\", \"DOM\": \"dominican republic\", \"DMA\": \"dominica\", \"DJI\": \"djibouti\", \"GRL\": \"greenland\", \"FRO\": \"faroe islands\", \"DNK\": \"denmark\", \"CZE\": \"czech republic\", \"CYN\": \"northern cyprus\", \"CYP\": \"cyprus\", \"CUB\": \"cuba\", \"HRV\": \"croatia\", \"CIV\": \"ivory coast\", \"CRI\": \"costa rica\", \"COD\": \"democratic republic of the congo\", \"COG\": \"republic of the congo\", \"COM\": \"comoros\", \"COL\": \"colombia\", \"CHN\": \"china\", \"MAC\": \"macao s.a.r\", \"HKG\": \"hong kong s.a.r.\", \"CHL\": \"chile\", \"TCD\": \"chad\", \"CAF\": \"central african republic\", \"CPV\": \"cape verde\", \"CAN\": \"canada\", \"CMR\": \"cameroon\", \"KHM\": \"cambodia\", \"MMR\": \"myanmar\", \"BDI\": \"burundi\", \"BFA\": \"burkina faso\", \"BGR\": \"bulgaria\", \"BRN\": \"brunei\", \"BRA\": \"brazil\", \"BWA\": \"botswana\", \"BIH\": \"bosnia and herzegovina\", \"BOL\": \"bolivia\", \"BTN\": \"bhutan\", \"BEN\": \"benin\", \"BLZ\": \"belize\", \"BEL\": \"belgium\", \"BLR\": \"belarus\", \"BRB\": \"barbados\", \"BGD\": \"bangladesh\", \"BHR\": \"bahrain\", \"BHS\": \"the bahamas\", \"AZE\": \"azerbaijan\", \"AUT\": \"austria\", \"AUS\": \"ashmore and cartier islands\", \"HMD\": \"heard island and mcdonald islands\", \"NFK\": \"norfolk island\", \"ARM\": \"armenia\", \"ARG\": \"argentina\", \"ATG\": \"antigua and barbuda\", \"AGO\": \"angola\", \"AND\": \"andorra\", \"DZA\": \"algeria\", \"ALB\": \"albania\", \"AFG\": \"afghanistan\", \"KAS\": \"siachen glacier\", \"ATA\": \"antarctica\", \"SXM\": \"sint maarten\", \"TUV\": \"tuvalu\"}")
    _STATE_TO_ABBR = json.loads("{\"Alabama\": \"AL\", \"Alaska\": \"AK\", \"Arizona\": \"AZ\", \"Arkansas\": \"AR\", \"California\": \"CA\", \"Colorado\": \"CO\", \"Connecticut\": \"CT\", \"Delaware\": \"DE\", \"Florida\": \"FL\", \"Georgia\": \"GA\", \"Hawaii\": \"HI\", \"Idaho\": \"ID\", \"Illinois\": \"IL\", \"Indiana\": \"IN\", \"Iowa\": \"IA\", \"Kansas\": \"KS\", \"Kentucky\": \"KY\", \"Louisiana\": \"LA\", \"Maine\": \"ME\", \"Maryland\": \"MD\", \"Massachusetts\": \"MA\", \"Michigan\": \"MI\", \"Minnesota\": \"MN\", \"Mississippi\": \"MS\", \"Missouri\": \"MO\", \"Montana\": \"MT\", \"Nebraska\": \"NE\", \"Nevada\": \"NV\", \"New Hampshire\": \"NH\", \"New Jersey\": \"NJ\", \"New Mexico\": \"NM\", \"New York\": \"NY\", \"North Carolina\": \"NC\", \"North Dakota\": \"ND\", \"Ohio\": \"OH\", \"Oklahoma\": \"OK\", \"Oregon\": \"OR\", \"Pennsylvania\": \"PA\", \"Rhode Island\": \"RI\", \"South Carolina\": \"SC\", \"South Dakota\": \"SD\", \"Tennessee\": \"TN\", \"Texas\": \"TX\", \"Utah\": \"UT\", \"Vermont\": \"VT\", \"Virginia\": \"VA\", \"Washington\": \"WA\", \"West Virginia\": \"WV\", \"Wisconsin\": \"WI\", \"Wyoming\": \"WY\", \"District of Columbia\": \"DC\"}")
    try:
        import plotly.express as px
        if level == 'world':
            # Detect whether the column holds ISO-3 codes (e.g. 'VEN', 'USA')
            # or full/partial country names -- handle both transparently.
            sample_vals = [str(r.get(name_col, '')).strip().upper() for r in rows[:10]]
            col_has_iso3 = sum(1 for v in sample_vals if v in _ISO3_TO_NAME) >= len(sample_vals) // 2
            for r in rows:
                raw = str(r.get(name_col, '')).strip()
                if col_has_iso3:
                    # Column already has ISO-3 codes -- use directly for Plotly;
                    # also set a display-friendly full lowercase name for hover.
                    iso3 = raw.upper()
                    r['_loc'] = iso3 if iso3 in _ISO3_TO_NAME else None
                    r['_display_name'] = _ISO3_TO_NAME.get(iso3, raw.lower())
                else:
                    # Column has country names -- look up the ISO-3 code.
                    r['_loc'] = _NAME_TO_ISO3.get(raw)
                    r['_display_name'] = raw.lower()
            hover_col = '_display_name'
            locationmode, scope = 'ISO-3', 'world'
        elif level == 'us_states':
            for r in rows:
                raw = str(r.get(name_col)).strip()
                r['_loc'] = _STATE_TO_ABBR.get(raw, raw.upper() if len(raw) == 2 else None)
                r['_display_name'] = raw.lower()
            hover_col = '_display_name'
            locationmode, scope = 'USA-states', 'usa'
        else:
            raise ValueError('this map used a custom .geojson -- Plotly has no built-in boundaries for it')
        mapped_rows = [r for r in rows if r.get('_loc')]
        if not mapped_rows:
            raise ValueError("none of the region names matched Plotly's built-in boundaries")
        fig = px.choropleth(
            mapped_rows, locations='_loc', locationmode=locationmode, color=value_col,
            hover_name=hover_col, color_continuous_scale='Viridis', scope=scope,
            title=chart.get('title') or 'AutoDS chart 1',
        )
        skipped = len(rows) - len(mapped_rows)
        if skipped:
            print(f'{skipped} region(s) had no matching Plotly boundary and were left blank on the map.')
        fig.show()
        _plotted = True
    except ImportError:
        print('plotly is not installed (pip install plotly) -- showing a ranked bar chart instead.')
    except Exception as _map_err:
        print(f'Could not build an interactive map ({_map_err}) -- showing a ranked bar chart instead.')
    if not _plotted:
        rows.sort(key=lambda r: r[value_col], reverse=True)
        labels = [str(r.get(name_col, '?')) for r in rows][:20][::-1]
        values = [r[value_col] for r in rows][:20][::-1]
        plt.barh(labels, values, color=chart.get('colors') or chart.get('color') or None)
        plt.xlabel(value_col)
elif x in df.columns:
    _vc = df[x].value_counts(dropna=True).head(15).sort_values()
    _label_colors = dict(zip(chart.get('labels') or [], chart.get('colors') or []))
    _bar_colors = [_label_colors.get(str(lbl), chart.get('color')) for lbl in _vc.index] if _label_colors else chart.get('color')
    _vc.plot(kind='barh', color=_bar_colors or None)
    plt.xlabel('Count')
else:
    print('The recorded chart columns are not present in the exported data.')

if not _plotted:
    plt.title(chart.get('title') or 'AutoDS chart 1')
    plt.tight_layout()
    plt.show()


**Your notes:**

The pie charts shows distribution of gender. Male(M) are the majority of the students with 74 out of 126 students being male. The number of females are 52. This  shows the difference between the two gender is 22

In [ ]:
import json
import matplotlib.pyplot as plt

chart = json.loads("{\"type\": \"boxplot\", \"x\": \"total_payment\", \"group\": \"gender\", \"x_label\": \"total_payment\", \"y_label\": \"total_payment\", \"caption\": \"Spread of 'total_payment': box = middle 50% of values, line = median, dots = outliers. Grouped by 'gender'.\", \"groups\": {\"M\": {\"min\": 0, \"q1\": 905, \"median\": 1025, \"q3\": 4812.94, \"max\": 5314.61, \"outliers\": [], \"sum\": 155049.07, \"count\": 74}, \"F\": {\"min\": 885.78, \"q1\": 1115, \"median\": 1300, \"q3\": 1400, \"max\": 1426.56, \"outliers\": [3363.75, 5314.61, 300, 374.4, 5314.61, 2527.24, 200, 5314.61, 2779.96, 200, 3057.95, 4070.14, 5314.61, 2297.49, 3700.12, 374.976], \"sum\": 88920.136, \"count\": 52}}, \"grouped\": true, \"insight\": \"Median 'total_payment' is 1,189.49, with the middle 50% of values between 950.00 and 2,041.15. 26 outlier(s) fall well outside that range.\", \"_boxplotMode\": \"simple\", \"title\": \"Sum of total_payment by gender\", \"reason\": \"\", \"_reportSource\": \"custom\", \"user_note\": \"this chart illustrates  total payment by gender. Male(M) studentts accounts for the highest total payment with $155041.1 compared to $88920.1 for female(F) students. This suggests that male customers contributes significantly more to total payments.\"}")
chart_type = chart.get('type')
x = chart.get('x')
y = chart.get('y')
_plotted = False  # set True below once a chart type renders via a non-matplotlib path

if chart_type in {'scatter', 'line'} and x in df.columns and y in df.columns:
    plot_df = df[[x, y]].dropna()
    if chart_type == 'line':
        plot_df = plot_df.sort_values(x)
        plt.plot(plot_df[x], plot_df[y], marker='o', color=chart.get('color') or None)
    else:
        plt.scatter(plot_df[x], plot_df[y], alpha=0.7, color=chart.get('color') or None)
    plt.xlabel(x)
    plt.ylabel(y)
elif chart_type == 'histogram' and x in df.columns:
    df[x].dropna().hist(bins=12, color=chart.get('color') or None)
    plt.xlabel(x)
elif chart_type == 'boxplot' and x in df.columns:
    # Replay using the SAME mode and grouping the person actually picked
    # (stored on the chart itself) -- 'simple' mode plots the sum per
    # group (matching a pivot table, with no quartiles/median/outliers),
    # 'full' mode replays an actual statistical box-and-whisker plot,
    # grouped the same way it was grouped live.
    _group_col = chart.get('group')
    _is_simple = chart.get('_boxplotMode') == 'simple'
    if _group_col and _group_col in df.columns:
        if _is_simple:
            _sums = df.groupby(_group_col)[x].sum().sort_values(ascending=False)
            plt.bar(_sums.index.astype(str), _sums.values, color=chart.get('color') or None)
            plt.ylabel(f'Sum of {x}')
            plt.xlabel(_group_col)
        else:
            df.boxplot(column=x, by=_group_col)
            plt.title('')
            plt.suptitle('')
            plt.ylabel(x)
            plt.xlabel(_group_col)
    elif _is_simple:
        _total = df[x].dropna().sum()
        plt.bar([x], [_total], color=chart.get('color') or None)
        plt.ylabel(f'Sum of {x}')
    else:
        df[x].dropna().plot(kind='box')
        plt.xlabel(x)
elif chart_type in {'bar', 'word_frequency'} and chart.get('labels'):
    labels = chart.get('labels', [])
    values = chart.get('values', [])
    _cs = chart.get('colors')
    plt.barh(labels[::-1], values[::-1], color=(_cs[::-1] if _cs else chart.get('color')) or None)
    plt.xlabel((chart.get('y_label') or chart.get('y')) if chart_type == 'bar' else 'Count')
elif chart_type == 'stacked_bar' and chart.get('series'):
    import pandas as _pd
    _labels = chart.get('labels', [])
    _stacked_df = _pd.DataFrame({s['label']: s['values'] for s in chart['series']}, index=_labels)
    _stacked_df.plot(kind='bar', stacked=True, ax=plt.gca())
    plt.xlabel(x)
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title=chart.get('group'), bbox_to_anchor=(1.02, 1), loc='upper left')
elif chart_type == 'wordcloud' and chart.get('words'):
    words = chart.get('words', [])
    labels = [w.get('word') for w in words][:25][::-1]
    values = [w.get('count') for w in words][:25][::-1]
    plt.barh(labels, values, color=chart.get('colors') or chart.get('color') or None)
    plt.xlabel('Count')
elif chart_type == 'choropleth' and chart.get('rows'):
    # Renders as an actual interactive map via Plotly Express, which ships
    # its own world/US-state boundaries -- no geopandas or shapefile needed.
    # Falls back to a static ranked bar chart if plotly isn't installed, or
    # if this map used a custom-uploaded .geojson whose shapes Plotly has no
    # built-in knowledge of.
    value_col = chart.get('value_col')
    name_col = chart.get('name_col')
    level = chart.get('level')
    rows = [r for r in chart['rows'] if r.get(value_col) is not None and r.get(name_col)]
    _NAME_TO_ISO3 = json.loads("{\"Zimbabwe\": \"ZWE\", \"Zambia\": \"ZMB\", \"Yemen\": \"YEM\", \"Vietnam\": \"VNM\", \"Venezuela\": \"VEN\", \"Vatican\": \"VAT\", \"Vanuatu\": \"VUT\", \"Uzbekistan\": \"UZB\", \"Uruguay\": \"URY\", \"Federated States of Micronesia\": \"FSM\", \"Marshall Islands\": \"MHL\", \"Northern Mariana Islands\": \"MNP\", \"United States Virgin Islands\": \"VIR\", \"Guam\": \"GUM\", \"American Samoa\": \"ASM\", \"Puerto Rico\": \"PRI\", \"United States of America\": \"USA\", \"South Georgia and the Islands\": \"SGS\", \"British Indian Ocean Territory\": \"IOT\", \"Saint Helena\": \"SHN\", \"Pitcairn Islands\": \"PCN\", \"Anguilla\": \"AIA\", \"Falkland Islands\": \"FLK\", \"Cayman Islands\": \"CYM\", \"Bermuda\": \"BMU\", \"British Virgin Islands\": \"VGB\", \"Turks and Caicos Islands\": \"TCA\", \"Montserrat\": \"MSR\", \"Jersey\": \"JEY\", \"Guernsey\": \"GGY\", \"Isle of Man\": \"IMN\", \"United Kingdom\": \"GBR\", \"United Arab Emirates\": \"ARE\", \"Ukraine\": \"UKR\", \"Uganda\": \"UGA\", \"Turkmenistan\": \"TKM\", \"Turkey\": \"TUR\", \"Tunisia\": \"TUN\", \"Trinidad and Tobago\": \"TTO\", \"Tonga\": \"TON\", \"Togo\": \"TGO\", \"East Timor\": \"TLS\", \"Thailand\": \"THA\", \"United Republic of Tanzania\": \"TZA\", \"Tajikistan\": \"TJK\", \"Taiwan\": \"TWN\", \"Syria\": \"SYR\", \"Switzerland\": \"CHE\", \"Sweden\": \"SWE\", \"Swaziland\": \"SWZ\", \"Suriname\": \"SUR\", \"South Sudan\": \"SSD\", \"Sudan\": \"SDN\", \"Sri Lanka\": \"LKA\", \"Spain\": \"ESP\", \"South Korea\": \"KOR\", \"South Africa\": \"ZAF\", \"Somalia\": \"SOM\", \"Somaliland\": \"SOL\", \"Solomon Islands\": \"SLB\", \"Slovakia\": \"SVK\", \"Slovenia\": \"SVN\", \"Singapore\": \"SGP\", \"Sierra Leone\": \"SLE\", \"Seychelles\": \"SYC\", \"Republic of Serbia\": \"SRB\", \"Senegal\": \"SEN\", \"Saudi Arabia\": \"SAU\", \"S\u00e3o Tom\u00e9 and Principe\": \"STP\", \"San Marino\": \"SMR\", \"Samoa\": \"WSM\", \"Saint Vincent and the Grenadines\": \"VCT\", \"Saint Lucia\": \"LCA\", \"Saint Kitts and Nevis\": \"KNA\", \"Rwanda\": \"RWA\", \"Russia\": \"RUS\", \"Romania\": \"ROU\", \"Qatar\": \"QAT\", \"Portugal\": \"PRT\", \"Poland\": \"POL\", \"Philippines\": \"PHL\", \"Peru\": \"PER\", \"Paraguay\": \"PRY\", \"Papua New Guinea\": \"PNG\", \"Panama\": \"PAN\", \"Palau\": \"PLW\", \"Pakistan\": \"PAK\", \"Oman\": \"OMN\", \"Norway\": \"NOR\", \"North Korea\": \"PRK\", \"Nigeria\": \"NGA\", \"Niger\": \"NER\", \"Nicaragua\": \"NIC\", \"New Zealand\": \"NZL\", \"Niue\": \"NIU\", \"Cook Islands\": \"COK\", \"Netherlands\": \"NLD\", \"Aruba\": \"ABW\", \"Cura\u00e7ao\": \"CUW\", \"Nepal\": \"NPL\", \"Nauru\": \"NRU\", \"Namibia\": \"NAM\", \"Mozambique\": \"MOZ\", \"Morocco\": \"MAR\", \"Western Sahara\": \"ESH\", \"Montenegro\": \"MNE\", \"Mongolia\": \"MNG\", \"Moldova\": \"MDA\", \"Monaco\": \"MCO\", \"Mexico\": \"MEX\", \"Mauritius\": \"MUS\", \"Mauritania\": \"MRT\", \"Malta\": \"MLT\", \"Mali\": \"MLI\", \"Maldives\": \"MDV\", \"Malaysia\": \"MYS\", \"Malawi\": \"MWI\", \"Madagascar\": \"MDG\", \"Macedonia\": \"MKD\", \"Luxembourg\": \"LUX\", \"Lithuania\": \"LTU\", \"Liechtenstein\": \"LIE\", \"Libya\": \"LBY\", \"Liberia\": \"LBR\", \"Lesotho\": \"LSO\", \"Lebanon\": \"LBN\", \"Latvia\": \"LVA\", \"Laos\": \"LAO\", \"Kyrgyzstan\": \"KGZ\", \"Kuwait\": \"KWT\", \"Kosovo\": \"KOS\", \"Kiribati\": \"KIR\", \"Kenya\": \"KEN\", \"Kazakhstan\": \"KAZ\", \"Jordan\": \"JOR\", \"Japan\": \"JPN\", \"Jamaica\": \"JAM\", \"Italy\": \"ITA\", \"Israel\": \"ISR\", \"Palestine\": \"PSE\", \"Ireland\": \"IRL\", \"Iraq\": \"IRQ\", \"Iran\": \"IRN\", \"Indonesia\": \"IDN\", \"India\": \"IND\", \"Iceland\": \"ISL\", \"Hungary\": \"HUN\", \"Honduras\": \"HND\", \"Haiti\": \"HTI\", \"Guyana\": \"GUY\", \"Guinea-Bissau\": \"GNB\", \"Guinea\": \"GIN\", \"Guatemala\": \"GTM\", \"Grenada\": \"GRD\", \"Greece\": \"GRC\", \"Ghana\": \"GHA\", \"Germany\": \"DEU\", \"Georgia\": \"GEO\", \"Gambia\": \"GMB\", \"Gabon\": \"GAB\", \"France\": \"FRA\", \"Saint Pierre and Miquelon\": \"SPM\", \"Wallis and Futuna\": \"WLF\", \"Saint Martin\": \"MAF\", \"Saint Barthelemy\": \"BLM\", \"French Polynesia\": \"PYF\", \"New Caledonia\": \"NCL\", \"French Southern and Antarctic Lands\": \"ATF\", \"Aland\": \"ALA\", \"Finland\": \"FIN\", \"Fiji\": \"FJI\", \"Ethiopia\": \"ETH\", \"Estonia\": \"EST\", \"Eritrea\": \"ERI\", \"Equatorial Guinea\": \"GNQ\", \"El Salvador\": \"SLV\", \"Egypt\": \"EGY\", \"Ecuador\": \"ECU\", \"Dominican Republic\": \"DOM\", \"Dominica\": \"DMA\", \"Djibouti\": \"DJI\", \"Greenland\": \"GRL\", \"Faroe Islands\": \"FRO\", \"Denmark\": \"DNK\", \"Czech Republic\": \"CZE\", \"Northern Cyprus\": \"CYN\", \"Cyprus\": \"CYP\", \"Cuba\": \"CUB\", \"Croatia\": \"HRV\", \"Ivory Coast\": \"CIV\", \"Costa Rica\": \"CRI\", \"Democratic Republic of the Congo\": \"COD\", \"Republic of the Congo\": \"COG\", \"Comoros\": \"COM\", \"Colombia\": \"COL\", \"China\": \"CHN\", \"Macao S.A.R\": \"MAC\", \"Hong Kong S.A.R.\": \"HKG\", \"Chile\": \"CHL\", \"Chad\": \"TCD\", \"Central African Republic\": \"CAF\", \"Cape Verde\": \"CPV\", \"Canada\": \"CAN\", \"Cameroon\": \"CMR\", \"Cambodia\": \"KHM\", \"Myanmar\": \"MMR\", \"Burundi\": \"BDI\", \"Burkina Faso\": \"BFA\", \"Bulgaria\": \"BGR\", \"Brunei\": \"BRN\", \"Brazil\": \"BRA\", \"Botswana\": \"BWA\", \"Bosnia and Herzegovina\": \"BIH\", \"Bolivia\": \"BOL\", \"Bhutan\": \"BTN\", \"Benin\": \"BEN\", \"Belize\": \"BLZ\", \"Belgium\": \"BEL\", \"Belarus\": \"BLR\", \"Barbados\": \"BRB\", \"Bangladesh\": \"BGD\", \"Bahrain\": \"BHR\", \"The Bahamas\": \"BHS\", \"Azerbaijan\": \"AZE\", \"Austria\": \"AUT\", \"Australia\": \"AUS\", \"Indian Ocean Territories\": \"AUS\", \"Heard Island and McDonald Islands\": \"HMD\", \"Norfolk Island\": \"NFK\", \"Ashmore and Cartier Islands\": \"AUS\", \"Armenia\": \"ARM\", \"Argentina\": \"ARG\", \"Antigua and Barbuda\": \"ATG\", \"Angola\": \"AGO\", \"Andorra\": \"AND\", \"Algeria\": \"DZA\", \"Albania\": \"ALB\", \"Afghanistan\": \"AFG\", \"Siachen Glacier\": \"KAS\", \"Antarctica\": \"ATA\", \"Sint Maarten\": \"SXM\", \"Tuvalu\": \"TUV\"}")
    _ISO3_TO_NAME = json.loads("{\"ZWE\": \"zimbabwe\", \"ZMB\": \"zambia\", \"YEM\": \"yemen\", \"VNM\": \"vietnam\", \"VEN\": \"venezuela\", \"VAT\": \"vatican\", \"VUT\": \"vanuatu\", \"UZB\": \"uzbekistan\", \"URY\": \"uruguay\", \"FSM\": \"federated states of micronesia\", \"MHL\": \"marshall islands\", \"MNP\": \"northern mariana islands\", \"VIR\": \"united states virgin islands\", \"GUM\": \"guam\", \"ASM\": \"american samoa\", \"PRI\": \"puerto rico\", \"USA\": \"united states of america\", \"SGS\": \"south georgia and the islands\", \"IOT\": \"british indian ocean territory\", \"SHN\": \"saint helena\", \"PCN\": \"pitcairn islands\", \"AIA\": \"anguilla\", \"FLK\": \"falkland islands\", \"CYM\": \"cayman islands\", \"BMU\": \"bermuda\", \"VGB\": \"british virgin islands\", \"TCA\": \"turks and caicos islands\", \"MSR\": \"montserrat\", \"JEY\": \"jersey\", \"GGY\": \"guernsey\", \"IMN\": \"isle of man\", \"GBR\": \"united kingdom\", \"ARE\": \"united arab emirates\", \"UKR\": \"ukraine\", \"UGA\": \"uganda\", \"TKM\": \"turkmenistan\", \"TUR\": \"turkey\", \"TUN\": \"tunisia\", \"TTO\": \"trinidad and tobago\", \"TON\": \"tonga\", \"TGO\": \"togo\", \"TLS\": \"east timor\", \"THA\": \"thailand\", \"TZA\": \"united republic of tanzania\", \"TJK\": \"tajikistan\", \"TWN\": \"taiwan\", \"SYR\": \"syria\", \"CHE\": \"switzerland\", \"SWE\": \"sweden\", \"SWZ\": \"swaziland\", \"SUR\": \"suriname\", \"SSD\": \"south sudan\", \"SDN\": \"sudan\", \"LKA\": \"sri lanka\", \"ESP\": \"spain\", \"KOR\": \"south korea\", \"ZAF\": \"south africa\", \"SOM\": \"somalia\", \"SOL\": \"somaliland\", \"SLB\": \"solomon islands\", \"SVK\": \"slovakia\", \"SVN\": \"slovenia\", \"SGP\": \"singapore\", \"SLE\": \"sierra leone\", \"SYC\": \"seychelles\", \"SRB\": \"republic of serbia\", \"SEN\": \"senegal\", \"SAU\": \"saudi arabia\", \"STP\": \"s\u00e3o tom\u00e9 and principe\", \"SMR\": \"san marino\", \"WSM\": \"samoa\", \"VCT\": \"saint vincent and the grenadines\", \"LCA\": \"saint lucia\", \"KNA\": \"saint kitts and nevis\", \"RWA\": \"rwanda\", \"RUS\": \"russia\", \"ROU\": \"romania\", \"QAT\": \"qatar\", \"PRT\": \"portugal\", \"POL\": \"poland\", \"PHL\": \"philippines\", \"PER\": \"peru\", \"PRY\": \"paraguay\", \"PNG\": \"papua new guinea\", \"PAN\": \"panama\", \"PLW\": \"palau\", \"PAK\": \"pakistan\", \"OMN\": \"oman\", \"NOR\": \"norway\", \"PRK\": \"north korea\", \"NGA\": \"nigeria\", \"NER\": \"niger\", \"NIC\": \"nicaragua\", \"NZL\": \"new zealand\", \"NIU\": \"niue\", \"COK\": \"cook islands\", \"NLD\": \"netherlands\", \"ABW\": \"aruba\", \"CUW\": \"cura\u00e7ao\", \"NPL\": \"nepal\", \"NRU\": \"nauru\", \"NAM\": \"namibia\", \"MOZ\": \"mozambique\", \"MAR\": \"morocco\", \"ESH\": \"western sahara\", \"MNE\": \"montenegro\", \"MNG\": \"mongolia\", \"MDA\": \"moldova\", \"MCO\": \"monaco\", \"MEX\": \"mexico\", \"MUS\": \"mauritius\", \"MRT\": \"mauritania\", \"MLT\": \"malta\", \"MLI\": \"mali\", \"MDV\": \"maldives\", \"MYS\": \"malaysia\", \"MWI\": \"malawi\", \"MDG\": \"madagascar\", \"MKD\": \"macedonia\", \"LUX\": \"luxembourg\", \"LTU\": \"lithuania\", \"LIE\": \"liechtenstein\", \"LBY\": \"libya\", \"LBR\": \"liberia\", \"LSO\": \"lesotho\", \"LBN\": \"lebanon\", \"LVA\": \"latvia\", \"LAO\": \"laos\", \"KGZ\": \"kyrgyzstan\", \"KWT\": \"kuwait\", \"KOS\": \"kosovo\", \"KIR\": \"kiribati\", \"KEN\": \"kenya\", \"KAZ\": \"kazakhstan\", \"JOR\": \"jordan\", \"JPN\": \"japan\", \"JAM\": \"jamaica\", \"ITA\": \"italy\", \"ISR\": \"israel\", \"PSE\": \"palestine\", \"IRL\": \"ireland\", \"IRQ\": \"iraq\", \"IRN\": \"iran\", \"IDN\": \"indonesia\", \"IND\": \"india\", \"ISL\": \"iceland\", \"HUN\": \"hungary\", \"HND\": \"honduras\", \"HTI\": \"haiti\", \"GUY\": \"guyana\", \"GNB\": \"guinea-bissau\", \"GIN\": \"guinea\", \"GTM\": \"guatemala\", \"GRD\": \"grenada\", \"GRC\": \"greece\", \"GHA\": \"ghana\", \"DEU\": \"germany\", \"GEO\": \"georgia\", \"GMB\": \"gambia\", \"GAB\": \"gabon\", \"FRA\": \"france\", \"SPM\": \"saint pierre and miquelon\", \"WLF\": \"wallis and futuna\", \"MAF\": \"saint martin\", \"BLM\": \"saint barthelemy\", \"PYF\": \"french polynesia\", \"NCL\": \"new caledonia\", \"ATF\": \"french southern and antarctic lands\", \"ALA\": \"aland\", \"FIN\": \"finland\", \"FJI\": \"fiji\", \"ETH\": \"ethiopia\", \"EST\": \"estonia\", \"ERI\": \"eritrea\", \"GNQ\": \"equatorial guinea\", \"SLV\": \"el salvador\", \"EGY\": \"egypt\", \"ECU\": \"ecuador\", \"DOM\": \"dominican republic\", \"DMA\": \"dominica\", \"DJI\": \"djibouti\", \"GRL\": \"greenland\", \"FRO\": \"faroe islands\", \"DNK\": \"denmark\", \"CZE\": \"czech republic\", \"CYN\": \"northern cyprus\", \"CYP\": \"cyprus\", \"CUB\": \"cuba\", \"HRV\": \"croatia\", \"CIV\": \"ivory coast\", \"CRI\": \"costa rica\", \"COD\": \"democratic republic of the congo\", \"COG\": \"republic of the congo\", \"COM\": \"comoros\", \"COL\": \"colombia\", \"CHN\": \"china\", \"MAC\": \"macao s.a.r\", \"HKG\": \"hong kong s.a.r.\", \"CHL\": \"chile\", \"TCD\": \"chad\", \"CAF\": \"central african republic\", \"CPV\": \"cape verde\", \"CAN\": \"canada\", \"CMR\": \"cameroon\", \"KHM\": \"cambodia\", \"MMR\": \"myanmar\", \"BDI\": \"burundi\", \"BFA\": \"burkina faso\", \"BGR\": \"bulgaria\", \"BRN\": \"brunei\", \"BRA\": \"brazil\", \"BWA\": \"botswana\", \"BIH\": \"bosnia and herzegovina\", \"BOL\": \"bolivia\", \"BTN\": \"bhutan\", \"BEN\": \"benin\", \"BLZ\": \"belize\", \"BEL\": \"belgium\", \"BLR\": \"belarus\", \"BRB\": \"barbados\", \"BGD\": \"bangladesh\", \"BHR\": \"bahrain\", \"BHS\": \"the bahamas\", \"AZE\": \"azerbaijan\", \"AUT\": \"austria\", \"AUS\": \"ashmore and cartier islands\", \"HMD\": \"heard island and mcdonald islands\", \"NFK\": \"norfolk island\", \"ARM\": \"armenia\", \"ARG\": \"argentina\", \"ATG\": \"antigua and barbuda\", \"AGO\": \"angola\", \"AND\": \"andorra\", \"DZA\": \"algeria\", \"ALB\": \"albania\", \"AFG\": \"afghanistan\", \"KAS\": \"siachen glacier\", \"ATA\": \"antarctica\", \"SXM\": \"sint maarten\", \"TUV\": \"tuvalu\"}")
    _STATE_TO_ABBR = json.loads("{\"Alabama\": \"AL\", \"Alaska\": \"AK\", \"Arizona\": \"AZ\", \"Arkansas\": \"AR\", \"California\": \"CA\", \"Colorado\": \"CO\", \"Connecticut\": \"CT\", \"Delaware\": \"DE\", \"Florida\": \"FL\", \"Georgia\": \"GA\", \"Hawaii\": \"HI\", \"Idaho\": \"ID\", \"Illinois\": \"IL\", \"Indiana\": \"IN\", \"Iowa\": \"IA\", \"Kansas\": \"KS\", \"Kentucky\": \"KY\", \"Louisiana\": \"LA\", \"Maine\": \"ME\", \"Maryland\": \"MD\", \"Massachusetts\": \"MA\", \"Michigan\": \"MI\", \"Minnesota\": \"MN\", \"Mississippi\": \"MS\", \"Missouri\": \"MO\", \"Montana\": \"MT\", \"Nebraska\": \"NE\", \"Nevada\": \"NV\", \"New Hampshire\": \"NH\", \"New Jersey\": \"NJ\", \"New Mexico\": \"NM\", \"New York\": \"NY\", \"North Carolina\": \"NC\", \"North Dakota\": \"ND\", \"Ohio\": \"OH\", \"Oklahoma\": \"OK\", \"Oregon\": \"OR\", \"Pennsylvania\": \"PA\", \"Rhode Island\": \"RI\", \"South Carolina\": \"SC\", \"South Dakota\": \"SD\", \"Tennessee\": \"TN\", \"Texas\": \"TX\", \"Utah\": \"UT\", \"Vermont\": \"VT\", \"Virginia\": \"VA\", \"Washington\": \"WA\", \"West Virginia\": \"WV\", \"Wisconsin\": \"WI\", \"Wyoming\": \"WY\", \"District of Columbia\": \"DC\"}")
    try:
        import plotly.express as px
        if level == 'world':
            # Detect whether the column holds ISO-3 codes (e.g. 'VEN', 'USA')
            # or full/partial country names -- handle both transparently.
            sample_vals = [str(r.get(name_col, '')).strip().upper() for r in rows[:10]]
            col_has_iso3 = sum(1 for v in sample_vals if v in _ISO3_TO_NAME) >= len(sample_vals) // 2
            for r in rows:
                raw = str(r.get(name_col, '')).strip()
                if col_has_iso3:
                    # Column already has ISO-3 codes -- use directly for Plotly;
                    # also set a display-friendly full lowercase name for hover.
                    iso3 = raw.upper()
                    r['_loc'] = iso3 if iso3 in _ISO3_TO_NAME else None
                    r['_display_name'] = _ISO3_TO_NAME.get(iso3, raw.lower())
                else:
                    # Column has country names -- look up the ISO-3 code.
                    r['_loc'] = _NAME_TO_ISO3.get(raw)
                    r['_display_name'] = raw.lower()
            hover_col = '_display_name'
            locationmode, scope = 'ISO-3', 'world'
        elif level == 'us_states':
            for r in rows:
                raw = str(r.get(name_col)).strip()
                r['_loc'] = _STATE_TO_ABBR.get(raw, raw.upper() if len(raw) == 2 else None)
                r['_display_name'] = raw.lower()
            hover_col = '_display_name'
            locationmode, scope = 'USA-states', 'usa'
        else:
            raise ValueError('this map used a custom .geojson -- Plotly has no built-in boundaries for it')
        mapped_rows = [r for r in rows if r.get('_loc')]
        if not mapped_rows:
            raise ValueError("none of the region names matched Plotly's built-in boundaries")
        fig = px.choropleth(
            mapped_rows, locations='_loc', locationmode=locationmode, color=value_col,
            hover_name=hover_col, color_continuous_scale='Viridis', scope=scope,
            title=chart.get('title') or 'AutoDS chart 2',
        )
        skipped = len(rows) - len(mapped_rows)
        if skipped:
            print(f'{skipped} region(s) had no matching Plotly boundary and were left blank on the map.')
        fig.show()
        _plotted = True
    except ImportError:
        print('plotly is not installed (pip install plotly) -- showing a ranked bar chart instead.')
    except Exception as _map_err:
        print(f'Could not build an interactive map ({_map_err}) -- showing a ranked bar chart instead.')
    if not _plotted:
        rows.sort(key=lambda r: r[value_col], reverse=True)
        labels = [str(r.get(name_col, '?')) for r in rows][:20][::-1]
        values = [r[value_col] for r in rows][:20][::-1]
        plt.barh(labels, values, color=chart.get('colors') or chart.get('color') or None)
        plt.xlabel(value_col)
elif x in df.columns:
    _vc = df[x].value_counts(dropna=True).head(15).sort_values()
    _label_colors = dict(zip(chart.get('labels') or [], chart.get('colors') or []))
    _bar_colors = [_label_colors.get(str(lbl), chart.get('color')) for lbl in _vc.index] if _label_colors else chart.get('color')
    _vc.plot(kind='barh', color=_bar_colors or None)
    plt.xlabel('Count')
else:
    print('The recorded chart columns are not present in the exported data.')

if not _plotted:
    plt.title(chart.get('title') or 'AutoDS chart 2')
    plt.tight_layout()
    plt.show()


**Your notes:**

this chart illustrates  total payment by gender. Male(M) studentts accounts for the highest total payment with $155041.1 compared to $88920.1 for female(F) students. This suggests that male customers contributes significantly more to total payments.

In [ ]:
import json
import matplotlib.pyplot as plt

chart = json.loads("{\"type\": \"pie\", \"x\": \"course\", \"caption\": \"Share of each category in 'course' \u2014 slice size = proportion of rows.\", \"labels\": [\"Data Science\", \"Machine Learning\", \"Web Development\", \"Data Analysis\", \"Cyber Security\"], \"values\": [39, 29, 25, 19, 14], \"insight\": \"'Data Science' is the most common value in 'course', accounting for 31.0% of 126 rows across 5 distinct value(s).\", \"title\": \"Share of course\", \"reason\": \"\", \"_reportSource\": \"custom\", \"user_note\": \"The charts show the distribution of enrolled course by counts.  Data Science is the most enrolled course with 39 of the 126 student enrolled.Machine learning is the second most enrolled course with 29 of the total students enrolled. Cyber Security is the least enrolled course with 14 enrollement which is a big gap compared to Data Science. Overall, the data suggest many students prefer Data Science compared to Cyber Security.\"}")
chart_type = chart.get('type')
x = chart.get('x')
y = chart.get('y')
_plotted = False  # set True below once a chart type renders via a non-matplotlib path

if chart_type in {'scatter', 'line'} and x in df.columns and y in df.columns:
    plot_df = df[[x, y]].dropna()
    if chart_type == 'line':
        plot_df = plot_df.sort_values(x)
        plt.plot(plot_df[x], plot_df[y], marker='o', color=chart.get('color') or None)
    else:
        plt.scatter(plot_df[x], plot_df[y], alpha=0.7, color=chart.get('color') or None)
    plt.xlabel(x)
    plt.ylabel(y)
elif chart_type == 'histogram' and x in df.columns:
    df[x].dropna().hist(bins=12, color=chart.get('color') or None)
    plt.xlabel(x)
elif chart_type == 'boxplot' and x in df.columns:
    # Replay using the SAME mode and grouping the person actually picked
    # (stored on the chart itself) -- 'simple' mode plots the sum per
    # group (matching a pivot table, with no quartiles/median/outliers),
    # 'full' mode replays an actual statistical box-and-whisker plot,
    # grouped the same way it was grouped live.
    _group_col = chart.get('group')
    _is_simple = chart.get('_boxplotMode') == 'simple'
    if _group_col and _group_col in df.columns:
        if _is_simple:
            _sums = df.groupby(_group_col)[x].sum().sort_values(ascending=False)
            plt.bar(_sums.index.astype(str), _sums.values, color=chart.get('color') or None)
            plt.ylabel(f'Sum of {x}')
            plt.xlabel(_group_col)
        else:
            df.boxplot(column=x, by=_group_col)
            plt.title('')
            plt.suptitle('')
            plt.ylabel(x)
            plt.xlabel(_group_col)
    elif _is_simple:
        _total = df[x].dropna().sum()
        plt.bar([x], [_total], color=chart.get('color') or None)
        plt.ylabel(f'Sum of {x}')
    else:
        df[x].dropna().plot(kind='box')
        plt.xlabel(x)
elif chart_type in {'bar', 'word_frequency'} and chart.get('labels'):
    labels = chart.get('labels', [])
    values = chart.get('values', [])
    _cs = chart.get('colors')
    plt.barh(labels[::-1], values[::-1], color=(_cs[::-1] if _cs else chart.get('color')) or None)
    plt.xlabel((chart.get('y_label') or chart.get('y')) if chart_type == 'bar' else 'Count')
elif chart_type == 'stacked_bar' and chart.get('series'):
    import pandas as _pd
    _labels = chart.get('labels', [])
    _stacked_df = _pd.DataFrame({s['label']: s['values'] for s in chart['series']}, index=_labels)
    _stacked_df.plot(kind='bar', stacked=True, ax=plt.gca())
    plt.xlabel(x)
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title=chart.get('group'), bbox_to_anchor=(1.02, 1), loc='upper left')
elif chart_type == 'wordcloud' and chart.get('words'):
    words = chart.get('words', [])
    labels = [w.get('word') for w in words][:25][::-1]
    values = [w.get('count') for w in words][:25][::-1]
    plt.barh(labels, values, color=chart.get('colors') or chart.get('color') or None)
    plt.xlabel('Count')
elif chart_type == 'choropleth' and chart.get('rows'):
    # Renders as an actual interactive map via Plotly Express, which ships
    # its own world/US-state boundaries -- no geopandas or shapefile needed.
    # Falls back to a static ranked bar chart if plotly isn't installed, or
    # if this map used a custom-uploaded .geojson whose shapes Plotly has no
    # built-in knowledge of.
    value_col = chart.get('value_col')
    name_col = chart.get('name_col')
    level = chart.get('level')
    rows = [r for r in chart['rows'] if r.get(value_col) is not None and r.get(name_col)]
    _NAME_TO_ISO3 = json.loads("{\"Zimbabwe\": \"ZWE\", \"Zambia\": \"ZMB\", \"Yemen\": \"YEM\", \"Vietnam\": \"VNM\", \"Venezuela\": \"VEN\", \"Vatican\": \"VAT\", \"Vanuatu\": \"VUT\", \"Uzbekistan\": \"UZB\", \"Uruguay\": \"URY\", \"Federated States of Micronesia\": \"FSM\", \"Marshall Islands\": \"MHL\", \"Northern Mariana Islands\": \"MNP\", \"United States Virgin Islands\": \"VIR\", \"Guam\": \"GUM\", \"American Samoa\": \"ASM\", \"Puerto Rico\": \"PRI\", \"United States of America\": \"USA\", \"South Georgia and the Islands\": \"SGS\", \"British Indian Ocean Territory\": \"IOT\", \"Saint Helena\": \"SHN\", \"Pitcairn Islands\": \"PCN\", \"Anguilla\": \"AIA\", \"Falkland Islands\": \"FLK\", \"Cayman Islands\": \"CYM\", \"Bermuda\": \"BMU\", \"British Virgin Islands\": \"VGB\", \"Turks and Caicos Islands\": \"TCA\", \"Montserrat\": \"MSR\", \"Jersey\": \"JEY\", \"Guernsey\": \"GGY\", \"Isle of Man\": \"IMN\", \"United Kingdom\": \"GBR\", \"United Arab Emirates\": \"ARE\", \"Ukraine\": \"UKR\", \"Uganda\": \"UGA\", \"Turkmenistan\": \"TKM\", \"Turkey\": \"TUR\", \"Tunisia\": \"TUN\", \"Trinidad and Tobago\": \"TTO\", \"Tonga\": \"TON\", \"Togo\": \"TGO\", \"East Timor\": \"TLS\", \"Thailand\": \"THA\", \"United Republic of Tanzania\": \"TZA\", \"Tajikistan\": \"TJK\", \"Taiwan\": \"TWN\", \"Syria\": \"SYR\", \"Switzerland\": \"CHE\", \"Sweden\": \"SWE\", \"Swaziland\": \"SWZ\", \"Suriname\": \"SUR\", \"South Sudan\": \"SSD\", \"Sudan\": \"SDN\", \"Sri Lanka\": \"LKA\", \"Spain\": \"ESP\", \"South Korea\": \"KOR\", \"South Africa\": \"ZAF\", \"Somalia\": \"SOM\", \"Somaliland\": \"SOL\", \"Solomon Islands\": \"SLB\", \"Slovakia\": \"SVK\", \"Slovenia\": \"SVN\", \"Singapore\": \"SGP\", \"Sierra Leone\": \"SLE\", \"Seychelles\": \"SYC\", \"Republic of Serbia\": \"SRB\", \"Senegal\": \"SEN\", \"Saudi Arabia\": \"SAU\", \"S\u00e3o Tom\u00e9 and Principe\": \"STP\", \"San Marino\": \"SMR\", \"Samoa\": \"WSM\", \"Saint Vincent and the Grenadines\": \"VCT\", \"Saint Lucia\": \"LCA\", \"Saint Kitts and Nevis\": \"KNA\", \"Rwanda\": \"RWA\", \"Russia\": \"RUS\", \"Romania\": \"ROU\", \"Qatar\": \"QAT\", \"Portugal\": \"PRT\", \"Poland\": \"POL\", \"Philippines\": \"PHL\", \"Peru\": \"PER\", \"Paraguay\": \"PRY\", \"Papua New Guinea\": \"PNG\", \"Panama\": \"PAN\", \"Palau\": \"PLW\", \"Pakistan\": \"PAK\", \"Oman\": \"OMN\", \"Norway\": \"NOR\", \"North Korea\": \"PRK\", \"Nigeria\": \"NGA\", \"Niger\": \"NER\", \"Nicaragua\": \"NIC\", \"New Zealand\": \"NZL\", \"Niue\": \"NIU\", \"Cook Islands\": \"COK\", \"Netherlands\": \"NLD\", \"Aruba\": \"ABW\", \"Cura\u00e7ao\": \"CUW\", \"Nepal\": \"NPL\", \"Nauru\": \"NRU\", \"Namibia\": \"NAM\", \"Mozambique\": \"MOZ\", \"Morocco\": \"MAR\", \"Western Sahara\": \"ESH\", \"Montenegro\": \"MNE\", \"Mongolia\": \"MNG\", \"Moldova\": \"MDA\", \"Monaco\": \"MCO\", \"Mexico\": \"MEX\", \"Mauritius\": \"MUS\", \"Mauritania\": \"MRT\", \"Malta\": \"MLT\", \"Mali\": \"MLI\", \"Maldives\": \"MDV\", \"Malaysia\": \"MYS\", \"Malawi\": \"MWI\", \"Madagascar\": \"MDG\", \"Macedonia\": \"MKD\", \"Luxembourg\": \"LUX\", \"Lithuania\": \"LTU\", \"Liechtenstein\": \"LIE\", \"Libya\": \"LBY\", \"Liberia\": \"LBR\", \"Lesotho\": \"LSO\", \"Lebanon\": \"LBN\", \"Latvia\": \"LVA\", \"Laos\": \"LAO\", \"Kyrgyzstan\": \"KGZ\", \"Kuwait\": \"KWT\", \"Kosovo\": \"KOS\", \"Kiribati\": \"KIR\", \"Kenya\": \"KEN\", \"Kazakhstan\": \"KAZ\", \"Jordan\": \"JOR\", \"Japan\": \"JPN\", \"Jamaica\": \"JAM\", \"Italy\": \"ITA\", \"Israel\": \"ISR\", \"Palestine\": \"PSE\", \"Ireland\": \"IRL\", \"Iraq\": \"IRQ\", \"Iran\": \"IRN\", \"Indonesia\": \"IDN\", \"India\": \"IND\", \"Iceland\": \"ISL\", \"Hungary\": \"HUN\", \"Honduras\": \"HND\", \"Haiti\": \"HTI\", \"Guyana\": \"GUY\", \"Guinea-Bissau\": \"GNB\", \"Guinea\": \"GIN\", \"Guatemala\": \"GTM\", \"Grenada\": \"GRD\", \"Greece\": \"GRC\", \"Ghana\": \"GHA\", \"Germany\": \"DEU\", \"Georgia\": \"GEO\", \"Gambia\": \"GMB\", \"Gabon\": \"GAB\", \"France\": \"FRA\", \"Saint Pierre and Miquelon\": \"SPM\", \"Wallis and Futuna\": \"WLF\", \"Saint Martin\": \"MAF\", \"Saint Barthelemy\": \"BLM\", \"French Polynesia\": \"PYF\", \"New Caledonia\": \"NCL\", \"French Southern and Antarctic Lands\": \"ATF\", \"Aland\": \"ALA\", \"Finland\": \"FIN\", \"Fiji\": \"FJI\", \"Ethiopia\": \"ETH\", \"Estonia\": \"EST\", \"Eritrea\": \"ERI\", \"Equatorial Guinea\": \"GNQ\", \"El Salvador\": \"SLV\", \"Egypt\": \"EGY\", \"Ecuador\": \"ECU\", \"Dominican Republic\": \"DOM\", \"Dominica\": \"DMA\", \"Djibouti\": \"DJI\", \"Greenland\": \"GRL\", \"Faroe Islands\": \"FRO\", \"Denmark\": \"DNK\", \"Czech Republic\": \"CZE\", \"Northern Cyprus\": \"CYN\", \"Cyprus\": \"CYP\", \"Cuba\": \"CUB\", \"Croatia\": \"HRV\", \"Ivory Coast\": \"CIV\", \"Costa Rica\": \"CRI\", \"Democratic Republic of the Congo\": \"COD\", \"Republic of the Congo\": \"COG\", \"Comoros\": \"COM\", \"Colombia\": \"COL\", \"China\": \"CHN\", \"Macao S.A.R\": \"MAC\", \"Hong Kong S.A.R.\": \"HKG\", \"Chile\": \"CHL\", \"Chad\": \"TCD\", \"Central African Republic\": \"CAF\", \"Cape Verde\": \"CPV\", \"Canada\": \"CAN\", \"Cameroon\": \"CMR\", \"Cambodia\": \"KHM\", \"Myanmar\": \"MMR\", \"Burundi\": \"BDI\", \"Burkina Faso\": \"BFA\", \"Bulgaria\": \"BGR\", \"Brunei\": \"BRN\", \"Brazil\": \"BRA\", \"Botswana\": \"BWA\", \"Bosnia and Herzegovina\": \"BIH\", \"Bolivia\": \"BOL\", \"Bhutan\": \"BTN\", \"Benin\": \"BEN\", \"Belize\": \"BLZ\", \"Belgium\": \"BEL\", \"Belarus\": \"BLR\", \"Barbados\": \"BRB\", \"Bangladesh\": \"BGD\", \"Bahrain\": \"BHR\", \"The Bahamas\": \"BHS\", \"Azerbaijan\": \"AZE\", \"Austria\": \"AUT\", \"Australia\": \"AUS\", \"Indian Ocean Territories\": \"AUS\", \"Heard Island and McDonald Islands\": \"HMD\", \"Norfolk Island\": \"NFK\", \"Ashmore and Cartier Islands\": \"AUS\", \"Armenia\": \"ARM\", \"Argentina\": \"ARG\", \"Antigua and Barbuda\": \"ATG\", \"Angola\": \"AGO\", \"Andorra\": \"AND\", \"Algeria\": \"DZA\", \"Albania\": \"ALB\", \"Afghanistan\": \"AFG\", \"Siachen Glacier\": \"KAS\", \"Antarctica\": \"ATA\", \"Sint Maarten\": \"SXM\", \"Tuvalu\": \"TUV\"}")
    _ISO3_TO_NAME = json.loads("{\"ZWE\": \"zimbabwe\", \"ZMB\": \"zambia\", \"YEM\": \"yemen\", \"VNM\": \"vietnam\", \"VEN\": \"venezuela\", \"VAT\": \"vatican\", \"VUT\": \"vanuatu\", \"UZB\": \"uzbekistan\", \"URY\": \"uruguay\", \"FSM\": \"federated states of micronesia\", \"MHL\": \"marshall islands\", \"MNP\": \"northern mariana islands\", \"VIR\": \"united states virgin islands\", \"GUM\": \"guam\", \"ASM\": \"american samoa\", \"PRI\": \"puerto rico\", \"USA\": \"united states of america\", \"SGS\": \"south georgia and the islands\", \"IOT\": \"british indian ocean territory\", \"SHN\": \"saint helena\", \"PCN\": \"pitcairn islands\", \"AIA\": \"anguilla\", \"FLK\": \"falkland islands\", \"CYM\": \"cayman islands\", \"BMU\": \"bermuda\", \"VGB\": \"british virgin islands\", \"TCA\": \"turks and caicos islands\", \"MSR\": \"montserrat\", \"JEY\": \"jersey\", \"GGY\": \"guernsey\", \"IMN\": \"isle of man\", \"GBR\": \"united kingdom\", \"ARE\": \"united arab emirates\", \"UKR\": \"ukraine\", \"UGA\": \"uganda\", \"TKM\": \"turkmenistan\", \"TUR\": \"turkey\", \"TUN\": \"tunisia\", \"TTO\": \"trinidad and tobago\", \"TON\": \"tonga\", \"TGO\": \"togo\", \"TLS\": \"east timor\", \"THA\": \"thailand\", \"TZA\": \"united republic of tanzania\", \"TJK\": \"tajikistan\", \"TWN\": \"taiwan\", \"SYR\": \"syria\", \"CHE\": \"switzerland\", \"SWE\": \"sweden\", \"SWZ\": \"swaziland\", \"SUR\": \"suriname\", \"SSD\": \"south sudan\", \"SDN\": \"sudan\", \"LKA\": \"sri lanka\", \"ESP\": \"spain\", \"KOR\": \"south korea\", \"ZAF\": \"south africa\", \"SOM\": \"somalia\", \"SOL\": \"somaliland\", \"SLB\": \"solomon islands\", \"SVK\": \"slovakia\", \"SVN\": \"slovenia\", \"SGP\": \"singapore\", \"SLE\": \"sierra leone\", \"SYC\": \"seychelles\", \"SRB\": \"republic of serbia\", \"SEN\": \"senegal\", \"SAU\": \"saudi arabia\", \"STP\": \"s\u00e3o tom\u00e9 and principe\", \"SMR\": \"san marino\", \"WSM\": \"samoa\", \"VCT\": \"saint vincent and the grenadines\", \"LCA\": \"saint lucia\", \"KNA\": \"saint kitts and nevis\", \"RWA\": \"rwanda\", \"RUS\": \"russia\", \"ROU\": \"romania\", \"QAT\": \"qatar\", \"PRT\": \"portugal\", \"POL\": \"poland\", \"PHL\": \"philippines\", \"PER\": \"peru\", \"PRY\": \"paraguay\", \"PNG\": \"papua new guinea\", \"PAN\": \"panama\", \"PLW\": \"palau\", \"PAK\": \"pakistan\", \"OMN\": \"oman\", \"NOR\": \"norway\", \"PRK\": \"north korea\", \"NGA\": \"nigeria\", \"NER\": \"niger\", \"NIC\": \"nicaragua\", \"NZL\": \"new zealand\", \"NIU\": \"niue\", \"COK\": \"cook islands\", \"NLD\": \"netherlands\", \"ABW\": \"aruba\", \"CUW\": \"cura\u00e7ao\", \"NPL\": \"nepal\", \"NRU\": \"nauru\", \"NAM\": \"namibia\", \"MOZ\": \"mozambique\", \"MAR\": \"morocco\", \"ESH\": \"western sahara\", \"MNE\": \"montenegro\", \"MNG\": \"mongolia\", \"MDA\": \"moldova\", \"MCO\": \"monaco\", \"MEX\": \"mexico\", \"MUS\": \"mauritius\", \"MRT\": \"mauritania\", \"MLT\": \"malta\", \"MLI\": \"mali\", \"MDV\": \"maldives\", \"MYS\": \"malaysia\", \"MWI\": \"malawi\", \"MDG\": \"madagascar\", \"MKD\": \"macedonia\", \"LUX\": \"luxembourg\", \"LTU\": \"lithuania\", \"LIE\": \"liechtenstein\", \"LBY\": \"libya\", \"LBR\": \"liberia\", \"LSO\": \"lesotho\", \"LBN\": \"lebanon\", \"LVA\": \"latvia\", \"LAO\": \"laos\", \"KGZ\": \"kyrgyzstan\", \"KWT\": \"kuwait\", \"KOS\": \"kosovo\", \"KIR\": \"kiribati\", \"KEN\": \"kenya\", \"KAZ\": \"kazakhstan\", \"JOR\": \"jordan\", \"JPN\": \"japan\", \"JAM\": \"jamaica\", \"ITA\": \"italy\", \"ISR\": \"israel\", \"PSE\": \"palestine\", \"IRL\": \"ireland\", \"IRQ\": \"iraq\", \"IRN\": \"iran\", \"IDN\": \"indonesia\", \"IND\": \"india\", \"ISL\": \"iceland\", \"HUN\": \"hungary\", \"HND\": \"honduras\", \"HTI\": \"haiti\", \"GUY\": \"guyana\", \"GNB\": \"guinea-bissau\", \"GIN\": \"guinea\", \"GTM\": \"guatemala\", \"GRD\": \"grenada\", \"GRC\": \"greece\", \"GHA\": \"ghana\", \"DEU\": \"germany\", \"GEO\": \"georgia\", \"GMB\": \"gambia\", \"GAB\": \"gabon\", \"FRA\": \"france\", \"SPM\": \"saint pierre and miquelon\", \"WLF\": \"wallis and futuna\", \"MAF\": \"saint martin\", \"BLM\": \"saint barthelemy\", \"PYF\": \"french polynesia\", \"NCL\": \"new caledonia\", \"ATF\": \"french southern and antarctic lands\", \"ALA\": \"aland\", \"FIN\": \"finland\", \"FJI\": \"fiji\", \"ETH\": \"ethiopia\", \"EST\": \"estonia\", \"ERI\": \"eritrea\", \"GNQ\": \"equatorial guinea\", \"SLV\": \"el salvador\", \"EGY\": \"egypt\", \"ECU\": \"ecuador\", \"DOM\": \"dominican republic\", \"DMA\": \"dominica\", \"DJI\": \"djibouti\", \"GRL\": \"greenland\", \"FRO\": \"faroe islands\", \"DNK\": \"denmark\", \"CZE\": \"czech republic\", \"CYN\": \"northern cyprus\", \"CYP\": \"cyprus\", \"CUB\": \"cuba\", \"HRV\": \"croatia\", \"CIV\": \"ivory coast\", \"CRI\": \"costa rica\", \"COD\": \"democratic republic of the congo\", \"COG\": \"republic of the congo\", \"COM\": \"comoros\", \"COL\": \"colombia\", \"CHN\": \"china\", \"MAC\": \"macao s.a.r\", \"HKG\": \"hong kong s.a.r.\", \"CHL\": \"chile\", \"TCD\": \"chad\", \"CAF\": \"central african republic\", \"CPV\": \"cape verde\", \"CAN\": \"canada\", \"CMR\": \"cameroon\", \"KHM\": \"cambodia\", \"MMR\": \"myanmar\", \"BDI\": \"burundi\", \"BFA\": \"burkina faso\", \"BGR\": \"bulgaria\", \"BRN\": \"brunei\", \"BRA\": \"brazil\", \"BWA\": \"botswana\", \"BIH\": \"bosnia and herzegovina\", \"BOL\": \"bolivia\", \"BTN\": \"bhutan\", \"BEN\": \"benin\", \"BLZ\": \"belize\", \"BEL\": \"belgium\", \"BLR\": \"belarus\", \"BRB\": \"barbados\", \"BGD\": \"bangladesh\", \"BHR\": \"bahrain\", \"BHS\": \"the bahamas\", \"AZE\": \"azerbaijan\", \"AUT\": \"austria\", \"AUS\": \"ashmore and cartier islands\", \"HMD\": \"heard island and mcdonald islands\", \"NFK\": \"norfolk island\", \"ARM\": \"armenia\", \"ARG\": \"argentina\", \"ATG\": \"antigua and barbuda\", \"AGO\": \"angola\", \"AND\": \"andorra\", \"DZA\": \"algeria\", \"ALB\": \"albania\", \"AFG\": \"afghanistan\", \"KAS\": \"siachen glacier\", \"ATA\": \"antarctica\", \"SXM\": \"sint maarten\", \"TUV\": \"tuvalu\"}")
    _STATE_TO_ABBR = json.loads("{\"Alabama\": \"AL\", \"Alaska\": \"AK\", \"Arizona\": \"AZ\", \"Arkansas\": \"AR\", \"California\": \"CA\", \"Colorado\": \"CO\", \"Connecticut\": \"CT\", \"Delaware\": \"DE\", \"Florida\": \"FL\", \"Georgia\": \"GA\", \"Hawaii\": \"HI\", \"Idaho\": \"ID\", \"Illinois\": \"IL\", \"Indiana\": \"IN\", \"Iowa\": \"IA\", \"Kansas\": \"KS\", \"Kentucky\": \"KY\", \"Louisiana\": \"LA\", \"Maine\": \"ME\", \"Maryland\": \"MD\", \"Massachusetts\": \"MA\", \"Michigan\": \"MI\", \"Minnesota\": \"MN\", \"Mississippi\": \"MS\", \"Missouri\": \"MO\", \"Montana\": \"MT\", \"Nebraska\": \"NE\", \"Nevada\": \"NV\", \"New Hampshire\": \"NH\", \"New Jersey\": \"NJ\", \"New Mexico\": \"NM\", \"New York\": \"NY\", \"North Carolina\": \"NC\", \"North Dakota\": \"ND\", \"Ohio\": \"OH\", \"Oklahoma\": \"OK\", \"Oregon\": \"OR\", \"Pennsylvania\": \"PA\", \"Rhode Island\": \"RI\", \"South Carolina\": \"SC\", \"South Dakota\": \"SD\", \"Tennessee\": \"TN\", \"Texas\": \"TX\", \"Utah\": \"UT\", \"Vermont\": \"VT\", \"Virginia\": \"VA\", \"Washington\": \"WA\", \"West Virginia\": \"WV\", \"Wisconsin\": \"WI\", \"Wyoming\": \"WY\", \"District of Columbia\": \"DC\"}")
    try:
        import plotly.express as px
        if level == 'world':
            # Detect whether the column holds ISO-3 codes (e.g. 'VEN', 'USA')
            # or full/partial country names -- handle both transparently.
            sample_vals = [str(r.get(name_col, '')).strip().upper() for r in rows[:10]]
            col_has_iso3 = sum(1 for v in sample_vals if v in _ISO3_TO_NAME) >= len(sample_vals) // 2
            for r in rows:
                raw = str(r.get(name_col, '')).strip()
                if col_has_iso3:
                    # Column already has ISO-3 codes -- use directly for Plotly;
                    # also set a display-friendly full lowercase name for hover.
                    iso3 = raw.upper()
                    r['_loc'] = iso3 if iso3 in _ISO3_TO_NAME else None
                    r['_display_name'] = _ISO3_TO_NAME.get(iso3, raw.lower())
                else:
                    # Column has country names -- look up the ISO-3 code.
                    r['_loc'] = _NAME_TO_ISO3.get(raw)
                    r['_display_name'] = raw.lower()
            hover_col = '_display_name'
            locationmode, scope = 'ISO-3', 'world'
        elif level == 'us_states':
            for r in rows:
                raw = str(r.get(name_col)).strip()
                r['_loc'] = _STATE_TO_ABBR.get(raw, raw.upper() if len(raw) == 2 else None)
                r['_display_name'] = raw.lower()
            hover_col = '_display_name'
            locationmode, scope = 'USA-states', 'usa'
        else:
            raise ValueError('this map used a custom .geojson -- Plotly has no built-in boundaries for it')
        mapped_rows = [r for r in rows if r.get('_loc')]
        if not mapped_rows:
            raise ValueError("none of the region names matched Plotly's built-in boundaries")
        fig = px.choropleth(
            mapped_rows, locations='_loc', locationmode=locationmode, color=value_col,
            hover_name=hover_col, color_continuous_scale='Viridis', scope=scope,
            title=chart.get('title') or 'AutoDS chart 3',
        )
        skipped = len(rows) - len(mapped_rows)
        if skipped:
            print(f'{skipped} region(s) had no matching Plotly boundary and were left blank on the map.')
        fig.show()
        _plotted = True
    except ImportError:
        print('plotly is not installed (pip install plotly) -- showing a ranked bar chart instead.')
    except Exception as _map_err:
        print(f'Could not build an interactive map ({_map_err}) -- showing a ranked bar chart instead.')
    if not _plotted:
        rows.sort(key=lambda r: r[value_col], reverse=True)
        labels = [str(r.get(name_col, '?')) for r in rows][:20][::-1]
        values = [r[value_col] for r in rows][:20][::-1]
        plt.barh(labels, values, color=chart.get('colors') or chart.get('color') or None)
        plt.xlabel(value_col)
elif x in df.columns:
    _vc = df[x].value_counts(dropna=True).head(15).sort_values()
    _label_colors = dict(zip(chart.get('labels') or [], chart.get('colors') or []))
    _bar_colors = [_label_colors.get(str(lbl), chart.get('color')) for lbl in _vc.index] if _label_colors else chart.get('color')
    _vc.plot(kind='barh', color=_bar_colors or None)
    plt.xlabel('Count')
else:
    print('The recorded chart columns are not present in the exported data.')

if not _plotted:
    plt.title(chart.get('title') or 'AutoDS chart 3')
    plt.tight_layout()
    plt.show()


**Your notes:**

The charts show the distribution of enrolled course by counts.  Data Science is the most enrolled course with 39 of the 126 student enrolled.Machine learning is the second most enrolled course with 29 of the total students enrolled. Cyber Security is the least enrolled course with 14 enrollement which is a big gap compared to Data Science. Overall, the data suggest many students prefer Data Science compared to Cyber Security.

In [ ]:
import json
import matplotlib.pyplot as plt

chart = json.loads("{\"type\": \"stacked_bar\", \"x\": \"age\", \"group\": \"course\", \"bar_limit\": 15, \"x_label\": \"age\", \"y_label\": \"Count\", \"caption\": \"Count of 'course' broken down within each 'age', stacked into one bar per 'age'. Showing top 15 of 16 'age' values.\", \"labels\": [\"23\", \"22\", \"21\", \"24\", \"33\", \"20\", \"25\", \"19\", \"26\", \"18\", \"28\", \"27\", \"29\", \"30\", \"31\"], \"series\": [{\"label\": \"Cyber Security\", \"values\": [3, 2, 1, 0, 4, 0, 0, 2, 1, 1, 0, 0, 0, 0, 0]}, {\"label\": \"Data Analysis\", \"values\": [8, 4, 1, 0, 2, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0]}, {\"label\": \"Data Science\", \"values\": [6, 8, 9, 5, 5, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0]}, {\"label\": \"Machine Learning\", \"values\": [3, 4, 5, 8, 1, 0, 5, 0, 0, 1, 0, 0, 0, 1, 0]}, {\"label\": \"Web Development\", \"values\": [4, 4, 1, 2, 2, 8, 1, 0, 0, 0, 1, 1, 0, 0, 1]}], \"title\": \"age by course (stacked)\", \"reason\": \"\", \"_reportSource\": \"custom\", \"user_note\": \"The chart shows the distribution of students across different age  by course. Students aged 23 and 22 have the highest representation, with age 23 having the largest number of students, followed closely by age 22. Data Science and Data Analysis contribute significantly to these age groups. Students aged 24 and 33 also show notable participation, particularly in Machine Learning and Data Science. Web Development appears across several age groups, including younger students and those in their late twenties and early thirties. However, the number of students decreases considerably among older age groups. Overall, the data suggests that most students are concentrated in their early twenties, although people of different ages are pursuing various technology courses.\"}")
chart_type = chart.get('type')
x = chart.get('x')
y = chart.get('y')
_plotted = False  # set True below once a chart type renders via a non-matplotlib path

if chart_type in {'scatter', 'line'} and x in df.columns and y in df.columns:
    plot_df = df[[x, y]].dropna()
    if chart_type == 'line':
        plot_df = plot_df.sort_values(x)
        plt.plot(plot_df[x], plot_df[y], marker='o', color=chart.get('color') or None)
    else:
        plt.scatter(plot_df[x], plot_df[y], alpha=0.7, color=chart.get('color') or None)
    plt.xlabel(x)
    plt.ylabel(y)
elif chart_type == 'histogram' and x in df.columns:
    df[x].dropna().hist(bins=12, color=chart.get('color') or None)
    plt.xlabel(x)
elif chart_type == 'boxplot' and x in df.columns:
    # Replay using the SAME mode and grouping the person actually picked
    # (stored on the chart itself) -- 'simple' mode plots the sum per
    # group (matching a pivot table, with no quartiles/median/outliers),
    # 'full' mode replays an actual statistical box-and-whisker plot,
    # grouped the same way it was grouped live.
    _group_col = chart.get('group')
    _is_simple = chart.get('_boxplotMode') == 'simple'
    if _group_col and _group_col in df.columns:
        if _is_simple:
            _sums = df.groupby(_group_col)[x].sum().sort_values(ascending=False)
            plt.bar(_sums.index.astype(str), _sums.values, color=chart.get('color') or None)
            plt.ylabel(f'Sum of {x}')
            plt.xlabel(_group_col)
        else:
            df.boxplot(column=x, by=_group_col)
            plt.title('')
            plt.suptitle('')
            plt.ylabel(x)
            plt.xlabel(_group_col)
    elif _is_simple:
        _total = df[x].dropna().sum()
        plt.bar([x], [_total], color=chart.get('color') or None)
        plt.ylabel(f'Sum of {x}')
    else:
        df[x].dropna().plot(kind='box')
        plt.xlabel(x)
elif chart_type in {'bar', 'word_frequency'} and chart.get('labels'):
    labels = chart.get('labels', [])
    values = chart.get('values', [])
    _cs = chart.get('colors')
    plt.barh(labels[::-1], values[::-1], color=(_cs[::-1] if _cs else chart.get('color')) or None)
    plt.xlabel((chart.get('y_label') or chart.get('y')) if chart_type == 'bar' else 'Count')
elif chart_type == 'stacked_bar' and chart.get('series'):
    import pandas as _pd
    _labels = chart.get('labels', [])
    _stacked_df = _pd.DataFrame({s['label']: s['values'] for s in chart['series']}, index=_labels)
    _stacked_df.plot(kind='bar', stacked=True, ax=plt.gca())
    plt.xlabel(x)
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title=chart.get('group'), bbox_to_anchor=(1.02, 1), loc='upper left')
elif chart_type == 'wordcloud' and chart.get('words'):
    words = chart.get('words', [])
    labels = [w.get('word') for w in words][:25][::-1]
    values = [w.get('count') for w in words][:25][::-1]
    plt.barh(labels, values, color=chart.get('colors') or chart.get('color') or None)
    plt.xlabel('Count')
elif chart_type == 'choropleth' and chart.get('rows'):
    # Renders as an actual interactive map via Plotly Express, which ships
    # its own world/US-state boundaries -- no geopandas or shapefile needed.
    # Falls back to a static ranked bar chart if plotly isn't installed, or
    # if this map used a custom-uploaded .geojson whose shapes Plotly has no
    # built-in knowledge of.
    value_col = chart.get('value_col')
    name_col = chart.get('name_col')
    level = chart.get('level')
    rows = [r for r in chart['rows'] if r.get(value_col) is not None and r.get(name_col)]
    _NAME_TO_ISO3 = json.loads("{\"Zimbabwe\": \"ZWE\", \"Zambia\": \"ZMB\", \"Yemen\": \"YEM\", \"Vietnam\": \"VNM\", \"Venezuela\": \"VEN\", \"Vatican\": \"VAT\", \"Vanuatu\": \"VUT\", \"Uzbekistan\": \"UZB\", \"Uruguay\": \"URY\", \"Federated States of Micronesia\": \"FSM\", \"Marshall Islands\": \"MHL\", \"Northern Mariana Islands\": \"MNP\", \"United States Virgin Islands\": \"VIR\", \"Guam\": \"GUM\", \"American Samoa\": \"ASM\", \"Puerto Rico\": \"PRI\", \"United States of America\": \"USA\", \"South Georgia and the Islands\": \"SGS\", \"British Indian Ocean Territory\": \"IOT\", \"Saint Helena\": \"SHN\", \"Pitcairn Islands\": \"PCN\", \"Anguilla\": \"AIA\", \"Falkland Islands\": \"FLK\", \"Cayman Islands\": \"CYM\", \"Bermuda\": \"BMU\", \"British Virgin Islands\": \"VGB\", \"Turks and Caicos Islands\": \"TCA\", \"Montserrat\": \"MSR\", \"Jersey\": \"JEY\", \"Guernsey\": \"GGY\", \"Isle of Man\": \"IMN\", \"United Kingdom\": \"GBR\", \"United Arab Emirates\": \"ARE\", \"Ukraine\": \"UKR\", \"Uganda\": \"UGA\", \"Turkmenistan\": \"TKM\", \"Turkey\": \"TUR\", \"Tunisia\": \"TUN\", \"Trinidad and Tobago\": \"TTO\", \"Tonga\": \"TON\", \"Togo\": \"TGO\", \"East Timor\": \"TLS\", \"Thailand\": \"THA\", \"United Republic of Tanzania\": \"TZA\", \"Tajikistan\": \"TJK\", \"Taiwan\": \"TWN\", \"Syria\": \"SYR\", \"Switzerland\": \"CHE\", \"Sweden\": \"SWE\", \"Swaziland\": \"SWZ\", \"Suriname\": \"SUR\", \"South Sudan\": \"SSD\", \"Sudan\": \"SDN\", \"Sri Lanka\": \"LKA\", \"Spain\": \"ESP\", \"South Korea\": \"KOR\", \"South Africa\": \"ZAF\", \"Somalia\": \"SOM\", \"Somaliland\": \"SOL\", \"Solomon Islands\": \"SLB\", \"Slovakia\": \"SVK\", \"Slovenia\": \"SVN\", \"Singapore\": \"SGP\", \"Sierra Leone\": \"SLE\", \"Seychelles\": \"SYC\", \"Republic of Serbia\": \"SRB\", \"Senegal\": \"SEN\", \"Saudi Arabia\": \"SAU\", \"S\u00e3o Tom\u00e9 and Principe\": \"STP\", \"San Marino\": \"SMR\", \"Samoa\": \"WSM\", \"Saint Vincent and the Grenadines\": \"VCT\", \"Saint Lucia\": \"LCA\", \"Saint Kitts and Nevis\": \"KNA\", \"Rwanda\": \"RWA\", \"Russia\": \"RUS\", \"Romania\": \"ROU\", \"Qatar\": \"QAT\", \"Portugal\": \"PRT\", \"Poland\": \"POL\", \"Philippines\": \"PHL\", \"Peru\": \"PER\", \"Paraguay\": \"PRY\", \"Papua New Guinea\": \"PNG\", \"Panama\": \"PAN\", \"Palau\": \"PLW\", \"Pakistan\": \"PAK\", \"Oman\": \"OMN\", \"Norway\": \"NOR\", \"North Korea\": \"PRK\", \"Nigeria\": \"NGA\", \"Niger\": \"NER\", \"Nicaragua\": \"NIC\", \"New Zealand\": \"NZL\", \"Niue\": \"NIU\", \"Cook Islands\": \"COK\", \"Netherlands\": \"NLD\", \"Aruba\": \"ABW\", \"Cura\u00e7ao\": \"CUW\", \"Nepal\": \"NPL\", \"Nauru\": \"NRU\", \"Namibia\": \"NAM\", \"Mozambique\": \"MOZ\", \"Morocco\": \"MAR\", \"Western Sahara\": \"ESH\", \"Montenegro\": \"MNE\", \"Mongolia\": \"MNG\", \"Moldova\": \"MDA\", \"Monaco\": \"MCO\", \"Mexico\": \"MEX\", \"Mauritius\": \"MUS\", \"Mauritania\": \"MRT\", \"Malta\": \"MLT\", \"Mali\": \"MLI\", \"Maldives\": \"MDV\", \"Malaysia\": \"MYS\", \"Malawi\": \"MWI\", \"Madagascar\": \"MDG\", \"Macedonia\": \"MKD\", \"Luxembourg\": \"LUX\", \"Lithuania\": \"LTU\", \"Liechtenstein\": \"LIE\", \"Libya\": \"LBY\", \"Liberia\": \"LBR\", \"Lesotho\": \"LSO\", \"Lebanon\": \"LBN\", \"Latvia\": \"LVA\", \"Laos\": \"LAO\", \"Kyrgyzstan\": \"KGZ\", \"Kuwait\": \"KWT\", \"Kosovo\": \"KOS\", \"Kiribati\": \"KIR\", \"Kenya\": \"KEN\", \"Kazakhstan\": \"KAZ\", \"Jordan\": \"JOR\", \"Japan\": \"JPN\", \"Jamaica\": \"JAM\", \"Italy\": \"ITA\", \"Israel\": \"ISR\", \"Palestine\": \"PSE\", \"Ireland\": \"IRL\", \"Iraq\": \"IRQ\", \"Iran\": \"IRN\", \"Indonesia\": \"IDN\", \"India\": \"IND\", \"Iceland\": \"ISL\", \"Hungary\": \"HUN\", \"Honduras\": \"HND\", \"Haiti\": \"HTI\", \"Guyana\": \"GUY\", \"Guinea-Bissau\": \"GNB\", \"Guinea\": \"GIN\", \"Guatemala\": \"GTM\", \"Grenada\": \"GRD\", \"Greece\": \"GRC\", \"Ghana\": \"GHA\", \"Germany\": \"DEU\", \"Georgia\": \"GEO\", \"Gambia\": \"GMB\", \"Gabon\": \"GAB\", \"France\": \"FRA\", \"Saint Pierre and Miquelon\": \"SPM\", \"Wallis and Futuna\": \"WLF\", \"Saint Martin\": \"MAF\", \"Saint Barthelemy\": \"BLM\", \"French Polynesia\": \"PYF\", \"New Caledonia\": \"NCL\", \"French Southern and Antarctic Lands\": \"ATF\", \"Aland\": \"ALA\", \"Finland\": \"FIN\", \"Fiji\": \"FJI\", \"Ethiopia\": \"ETH\", \"Estonia\": \"EST\", \"Eritrea\": \"ERI\", \"Equatorial Guinea\": \"GNQ\", \"El Salvador\": \"SLV\", \"Egypt\": \"EGY\", \"Ecuador\": \"ECU\", \"Dominican Republic\": \"DOM\", \"Dominica\": \"DMA\", \"Djibouti\": \"DJI\", \"Greenland\": \"GRL\", \"Faroe Islands\": \"FRO\", \"Denmark\": \"DNK\", \"Czech Republic\": \"CZE\", \"Northern Cyprus\": \"CYN\", \"Cyprus\": \"CYP\", \"Cuba\": \"CUB\", \"Croatia\": \"HRV\", \"Ivory Coast\": \"CIV\", \"Costa Rica\": \"CRI\", \"Democratic Republic of the Congo\": \"COD\", \"Republic of the Congo\": \"COG\", \"Comoros\": \"COM\", \"Colombia\": \"COL\", \"China\": \"CHN\", \"Macao S.A.R\": \"MAC\", \"Hong Kong S.A.R.\": \"HKG\", \"Chile\": \"CHL\", \"Chad\": \"TCD\", \"Central African Republic\": \"CAF\", \"Cape Verde\": \"CPV\", \"Canada\": \"CAN\", \"Cameroon\": \"CMR\", \"Cambodia\": \"KHM\", \"Myanmar\": \"MMR\", \"Burundi\": \"BDI\", \"Burkina Faso\": \"BFA\", \"Bulgaria\": \"BGR\", \"Brunei\": \"BRN\", \"Brazil\": \"BRA\", \"Botswana\": \"BWA\", \"Bosnia and Herzegovina\": \"BIH\", \"Bolivia\": \"BOL\", \"Bhutan\": \"BTN\", \"Benin\": \"BEN\", \"Belize\": \"BLZ\", \"Belgium\": \"BEL\", \"Belarus\": \"BLR\", \"Barbados\": \"BRB\", \"Bangladesh\": \"BGD\", \"Bahrain\": \"BHR\", \"The Bahamas\": \"BHS\", \"Azerbaijan\": \"AZE\", \"Austria\": \"AUT\", \"Australia\": \"AUS\", \"Indian Ocean Territories\": \"AUS\", \"Heard Island and McDonald Islands\": \"HMD\", \"Norfolk Island\": \"NFK\", \"Ashmore and Cartier Islands\": \"AUS\", \"Armenia\": \"ARM\", \"Argentina\": \"ARG\", \"Antigua and Barbuda\": \"ATG\", \"Angola\": \"AGO\", \"Andorra\": \"AND\", \"Algeria\": \"DZA\", \"Albania\": \"ALB\", \"Afghanistan\": \"AFG\", \"Siachen Glacier\": \"KAS\", \"Antarctica\": \"ATA\", \"Sint Maarten\": \"SXM\", \"Tuvalu\": \"TUV\"}")
    _ISO3_TO_NAME = json.loads("{\"ZWE\": \"zimbabwe\", \"ZMB\": \"zambia\", \"YEM\": \"yemen\", \"VNM\": \"vietnam\", \"VEN\": \"venezuela\", \"VAT\": \"vatican\", \"VUT\": \"vanuatu\", \"UZB\": \"uzbekistan\", \"URY\": \"uruguay\", \"FSM\": \"federated states of micronesia\", \"MHL\": \"marshall islands\", \"MNP\": \"northern mariana islands\", \"VIR\": \"united states virgin islands\", \"GUM\": \"guam\", \"ASM\": \"american samoa\", \"PRI\": \"puerto rico\", \"USA\": \"united states of america\", \"SGS\": \"south georgia and the islands\", \"IOT\": \"british indian ocean territory\", \"SHN\": \"saint helena\", \"PCN\": \"pitcairn islands\", \"AIA\": \"anguilla\", \"FLK\": \"falkland islands\", \"CYM\": \"cayman islands\", \"BMU\": \"bermuda\", \"VGB\": \"british virgin islands\", \"TCA\": \"turks and caicos islands\", \"MSR\": \"montserrat\", \"JEY\": \"jersey\", \"GGY\": \"guernsey\", \"IMN\": \"isle of man\", \"GBR\": \"united kingdom\", \"ARE\": \"united arab emirates\", \"UKR\": \"ukraine\", \"UGA\": \"uganda\", \"TKM\": \"turkmenistan\", \"TUR\": \"turkey\", \"TUN\": \"tunisia\", \"TTO\": \"trinidad and tobago\", \"TON\": \"tonga\", \"TGO\": \"togo\", \"TLS\": \"east timor\", \"THA\": \"thailand\", \"TZA\": \"united republic of tanzania\", \"TJK\": \"tajikistan\", \"TWN\": \"taiwan\", \"SYR\": \"syria\", \"CHE\": \"switzerland\", \"SWE\": \"sweden\", \"SWZ\": \"swaziland\", \"SUR\": \"suriname\", \"SSD\": \"south sudan\", \"SDN\": \"sudan\", \"LKA\": \"sri lanka\", \"ESP\": \"spain\", \"KOR\": \"south korea\", \"ZAF\": \"south africa\", \"SOM\": \"somalia\", \"SOL\": \"somaliland\", \"SLB\": \"solomon islands\", \"SVK\": \"slovakia\", \"SVN\": \"slovenia\", \"SGP\": \"singapore\", \"SLE\": \"sierra leone\", \"SYC\": \"seychelles\", \"SRB\": \"republic of serbia\", \"SEN\": \"senegal\", \"SAU\": \"saudi arabia\", \"STP\": \"s\u00e3o tom\u00e9 and principe\", \"SMR\": \"san marino\", \"WSM\": \"samoa\", \"VCT\": \"saint vincent and the grenadines\", \"LCA\": \"saint lucia\", \"KNA\": \"saint kitts and nevis\", \"RWA\": \"rwanda\", \"RUS\": \"russia\", \"ROU\": \"romania\", \"QAT\": \"qatar\", \"PRT\": \"portugal\", \"POL\": \"poland\", \"PHL\": \"philippines\", \"PER\": \"peru\", \"PRY\": \"paraguay\", \"PNG\": \"papua new guinea\", \"PAN\": \"panama\", \"PLW\": \"palau\", \"PAK\": \"pakistan\", \"OMN\": \"oman\", \"NOR\": \"norway\", \"PRK\": \"north korea\", \"NGA\": \"nigeria\", \"NER\": \"niger\", \"NIC\": \"nicaragua\", \"NZL\": \"new zealand\", \"NIU\": \"niue\", \"COK\": \"cook islands\", \"NLD\": \"netherlands\", \"ABW\": \"aruba\", \"CUW\": \"cura\u00e7ao\", \"NPL\": \"nepal\", \"NRU\": \"nauru\", \"NAM\": \"namibia\", \"MOZ\": \"mozambique\", \"MAR\": \"morocco\", \"ESH\": \"western sahara\", \"MNE\": \"montenegro\", \"MNG\": \"mongolia\", \"MDA\": \"moldova\", \"MCO\": \"monaco\", \"MEX\": \"mexico\", \"MUS\": \"mauritius\", \"MRT\": \"mauritania\", \"MLT\": \"malta\", \"MLI\": \"mali\", \"MDV\": \"maldives\", \"MYS\": \"malaysia\", \"MWI\": \"malawi\", \"MDG\": \"madagascar\", \"MKD\": \"macedonia\", \"LUX\": \"luxembourg\", \"LTU\": \"lithuania\", \"LIE\": \"liechtenstein\", \"LBY\": \"libya\", \"LBR\": \"liberia\", \"LSO\": \"lesotho\", \"LBN\": \"lebanon\", \"LVA\": \"latvia\", \"LAO\": \"laos\", \"KGZ\": \"kyrgyzstan\", \"KWT\": \"kuwait\", \"KOS\": \"kosovo\", \"KIR\": \"kiribati\", \"KEN\": \"kenya\", \"KAZ\": \"kazakhstan\", \"JOR\": \"jordan\", \"JPN\": \"japan\", \"JAM\": \"jamaica\", \"ITA\": \"italy\", \"ISR\": \"israel\", \"PSE\": \"palestine\", \"IRL\": \"ireland\", \"IRQ\": \"iraq\", \"IRN\": \"iran\", \"IDN\": \"indonesia\", \"IND\": \"india\", \"ISL\": \"iceland\", \"HUN\": \"hungary\", \"HND\": \"honduras\", \"HTI\": \"haiti\", \"GUY\": \"guyana\", \"GNB\": \"guinea-bissau\", \"GIN\": \"guinea\", \"GTM\": \"guatemala\", \"GRD\": \"grenada\", \"GRC\": \"greece\", \"GHA\": \"ghana\", \"DEU\": \"germany\", \"GEO\": \"georgia\", \"GMB\": \"gambia\", \"GAB\": \"gabon\", \"FRA\": \"france\", \"SPM\": \"saint pierre and miquelon\", \"WLF\": \"wallis and futuna\", \"MAF\": \"saint martin\", \"BLM\": \"saint barthelemy\", \"PYF\": \"french polynesia\", \"NCL\": \"new caledonia\", \"ATF\": \"french southern and antarctic lands\", \"ALA\": \"aland\", \"FIN\": \"finland\", \"FJI\": \"fiji\", \"ETH\": \"ethiopia\", \"EST\": \"estonia\", \"ERI\": \"eritrea\", \"GNQ\": \"equatorial guinea\", \"SLV\": \"el salvador\", \"EGY\": \"egypt\", \"ECU\": \"ecuador\", \"DOM\": \"dominican republic\", \"DMA\": \"dominica\", \"DJI\": \"djibouti\", \"GRL\": \"greenland\", \"FRO\": \"faroe islands\", \"DNK\": \"denmark\", \"CZE\": \"czech republic\", \"CYN\": \"northern cyprus\", \"CYP\": \"cyprus\", \"CUB\": \"cuba\", \"HRV\": \"croatia\", \"CIV\": \"ivory coast\", \"CRI\": \"costa rica\", \"COD\": \"democratic republic of the congo\", \"COG\": \"republic of the congo\", \"COM\": \"comoros\", \"COL\": \"colombia\", \"CHN\": \"china\", \"MAC\": \"macao s.a.r\", \"HKG\": \"hong kong s.a.r.\", \"CHL\": \"chile\", \"TCD\": \"chad\", \"CAF\": \"central african republic\", \"CPV\": \"cape verde\", \"CAN\": \"canada\", \"CMR\": \"cameroon\", \"KHM\": \"cambodia\", \"MMR\": \"myanmar\", \"BDI\": \"burundi\", \"BFA\": \"burkina faso\", \"BGR\": \"bulgaria\", \"BRN\": \"brunei\", \"BRA\": \"brazil\", \"BWA\": \"botswana\", \"BIH\": \"bosnia and herzegovina\", \"BOL\": \"bolivia\", \"BTN\": \"bhutan\", \"BEN\": \"benin\", \"BLZ\": \"belize\", \"BEL\": \"belgium\", \"BLR\": \"belarus\", \"BRB\": \"barbados\", \"BGD\": \"bangladesh\", \"BHR\": \"bahrain\", \"BHS\": \"the bahamas\", \"AZE\": \"azerbaijan\", \"AUT\": \"austria\", \"AUS\": \"ashmore and cartier islands\", \"HMD\": \"heard island and mcdonald islands\", \"NFK\": \"norfolk island\", \"ARM\": \"armenia\", \"ARG\": \"argentina\", \"ATG\": \"antigua and barbuda\", \"AGO\": \"angola\", \"AND\": \"andorra\", \"DZA\": \"algeria\", \"ALB\": \"albania\", \"AFG\": \"afghanistan\", \"KAS\": \"siachen glacier\", \"ATA\": \"antarctica\", \"SXM\": \"sint maarten\", \"TUV\": \"tuvalu\"}")
    _STATE_TO_ABBR = json.loads("{\"Alabama\": \"AL\", \"Alaska\": \"AK\", \"Arizona\": \"AZ\", \"Arkansas\": \"AR\", \"California\": \"CA\", \"Colorado\": \"CO\", \"Connecticut\": \"CT\", \"Delaware\": \"DE\", \"Florida\": \"FL\", \"Georgia\": \"GA\", \"Hawaii\": \"HI\", \"Idaho\": \"ID\", \"Illinois\": \"IL\", \"Indiana\": \"IN\", \"Iowa\": \"IA\", \"Kansas\": \"KS\", \"Kentucky\": \"KY\", \"Louisiana\": \"LA\", \"Maine\": \"ME\", \"Maryland\": \"MD\", \"Massachusetts\": \"MA\", \"Michigan\": \"MI\", \"Minnesota\": \"MN\", \"Mississippi\": \"MS\", \"Missouri\": \"MO\", \"Montana\": \"MT\", \"Nebraska\": \"NE\", \"Nevada\": \"NV\", \"New Hampshire\": \"NH\", \"New Jersey\": \"NJ\", \"New Mexico\": \"NM\", \"New York\": \"NY\", \"North Carolina\": \"NC\", \"North Dakota\": \"ND\", \"Ohio\": \"OH\", \"Oklahoma\": \"OK\", \"Oregon\": \"OR\", \"Pennsylvania\": \"PA\", \"Rhode Island\": \"RI\", \"South Carolina\": \"SC\", \"South Dakota\": \"SD\", \"Tennessee\": \"TN\", \"Texas\": \"TX\", \"Utah\": \"UT\", \"Vermont\": \"VT\", \"Virginia\": \"VA\", \"Washington\": \"WA\", \"West Virginia\": \"WV\", \"Wisconsin\": \"WI\", \"Wyoming\": \"WY\", \"District of Columbia\": \"DC\"}")
    try:
        import plotly.express as px
        if level == 'world':
            # Detect whether the column holds ISO-3 codes (e.g. 'VEN', 'USA')
            # or full/partial country names -- handle both transparently.
            sample_vals = [str(r.get(name_col, '')).strip().upper() for r in rows[:10]]
            col_has_iso3 = sum(1 for v in sample_vals if v in _ISO3_TO_NAME) >= len(sample_vals) // 2
            for r in rows:
                raw = str(r.get(name_col, '')).strip()
                if col_has_iso3:
                    # Column already has ISO-3 codes -- use directly for Plotly;
                    # also set a display-friendly full lowercase name for hover.
                    iso3 = raw.upper()
                    r['_loc'] = iso3 if iso3 in _ISO3_TO_NAME else None
                    r['_display_name'] = _ISO3_TO_NAME.get(iso3, raw.lower())
                else:
                    # Column has country names -- look up the ISO-3 code.
                    r['_loc'] = _NAME_TO_ISO3.get(raw)
                    r['_display_name'] = raw.lower()
            hover_col = '_display_name'
            locationmode, scope = 'ISO-3', 'world'
        elif level == 'us_states':
            for r in rows:
                raw = str(r.get(name_col)).strip()
                r['_loc'] = _STATE_TO_ABBR.get(raw, raw.upper() if len(raw) == 2 else None)
                r['_display_name'] = raw.lower()
            hover_col = '_display_name'
            locationmode, scope = 'USA-states', 'usa'
        else:
            raise ValueError('this map used a custom .geojson -- Plotly has no built-in boundaries for it')
        mapped_rows = [r for r in rows if r.get('_loc')]
        if not mapped_rows:
            raise ValueError("none of the region names matched Plotly's built-in boundaries")
        fig = px.choropleth(
            mapped_rows, locations='_loc', locationmode=locationmode, color=value_col,
            hover_name=hover_col, color_continuous_scale='Viridis', scope=scope,
            title=chart.get('title') or 'AutoDS chart 4',
        )
        skipped = len(rows) - len(mapped_rows)
        if skipped:
            print(f'{skipped} region(s) had no matching Plotly boundary and were left blank on the map.')
        fig.show()
        _plotted = True
    except ImportError:
        print('plotly is not installed (pip install plotly) -- showing a ranked bar chart instead.')
    except Exception as _map_err:
        print(f'Could not build an interactive map ({_map_err}) -- showing a ranked bar chart instead.')
    if not _plotted:
        rows.sort(key=lambda r: r[value_col], reverse=True)
        labels = [str(r.get(name_col, '?')) for r in rows][:20][::-1]
        values = [r[value_col] for r in rows][:20][::-1]
        plt.barh(labels, values, color=chart.get('colors') or chart.get('color') or None)
        plt.xlabel(value_col)
elif x in df.columns:
    _vc = df[x].value_counts(dropna=True).head(15).sort_values()
    _label_colors = dict(zip(chart.get('labels') or [], chart.get('colors') or []))
    _bar_colors = [_label_colors.get(str(lbl), chart.get('color')) for lbl in _vc.index] if _label_colors else chart.get('color')
    _vc.plot(kind='barh', color=_bar_colors or None)
    plt.xlabel('Count')
else:
    print('The recorded chart columns are not present in the exported data.')

if not _plotted:
    plt.title(chart.get('title') or 'AutoDS chart 4')
    plt.tight_layout()
    plt.show()


**Your notes:**

The chart shows the distribution of students across different age  by course. Students aged 23 and 22 have the highest representation, with age 23 having the largest number of students, followed closely by age 22. Data Science and Data Analysis contribute significantly to these age groups. Students aged 24 and 33 also show notable participation, particularly in Machine Learning and Data Science. Web Development appears across several age groups, including younger students and those in their late twenties and early thirties. However, the number of students decreases considerably among older age groups. Overall, the data suggests that most students are concentrated in their early twenties, although people of different ages are pursuing various technology courses.

In [ ]:
import json
import matplotlib.pyplot as plt

chart = json.loads("{\"type\": \"stacked_bar\", \"x\": \"course\", \"group\": \"gender\", \"bar_limit\": 15, \"x_label\": \"course\", \"y_label\": \"Count\", \"caption\": \"Count of 'gender' broken down within each 'course', stacked into one bar per 'course'.\", \"labels\": [\"Data Science\", \"Machine Learning\", \"Web Development\", \"Data Analysis\", \"Cyber Security\"], \"series\": [{\"label\": \"F\", \"values\": [12, 27, 3, 4, 6]}, {\"label\": \"M\", \"values\": [27, 2, 22, 15, 8]}], \"title\": \"course by gender (stacked)\", \"reason\": \"\", \"_reportSource\": \"custom\", \"user_note\": \"from the chart, in females(F) Macine Learing is the most enrolled course with 27 out of the 52 students enrolled while in Males(M) it is the least enrolled course with only 2 entries.\\nData science is the one prefered by most Males with 27 male students enrolled. Web Develpoment is also preffered among male students with 22 enrolled male. \\ncyber security is the least enrolled course generally\"}")
chart_type = chart.get('type')
x = chart.get('x')
y = chart.get('y')
_plotted = False  # set True below once a chart type renders via a non-matplotlib path

if chart_type in {'scatter', 'line'} and x in df.columns and y in df.columns:
    plot_df = df[[x, y]].dropna()
    if chart_type == 'line':
        plot_df = plot_df.sort_values(x)
        plt.plot(plot_df[x], plot_df[y], marker='o', color=chart.get('color') or None)
    else:
        plt.scatter(plot_df[x], plot_df[y], alpha=0.7, color=chart.get('color') or None)
    plt.xlabel(x)
    plt.ylabel(y)
elif chart_type == 'histogram' and x in df.columns:
    df[x].dropna().hist(bins=12, color=chart.get('color') or None)
    plt.xlabel(x)
elif chart_type == 'boxplot' and x in df.columns:
    # Replay using the SAME mode and grouping the person actually picked
    # (stored on the chart itself) -- 'simple' mode plots the sum per
    # group (matching a pivot table, with no quartiles/median/outliers),
    # 'full' mode replays an actual statistical box-and-whisker plot,
    # grouped the same way it was grouped live.
    _group_col = chart.get('group')
    _is_simple = chart.get('_boxplotMode') == 'simple'
    if _group_col and _group_col in df.columns:
        if _is_simple:
            _sums = df.groupby(_group_col)[x].sum().sort_values(ascending=False)
            plt.bar(_sums.index.astype(str), _sums.values, color=chart.get('color') or None)
            plt.ylabel(f'Sum of {x}')
            plt.xlabel(_group_col)
        else:
            df.boxplot(column=x, by=_group_col)
            plt.title('')
            plt.suptitle('')
            plt.ylabel(x)
            plt.xlabel(_group_col)
    elif _is_simple:
        _total = df[x].dropna().sum()
        plt.bar([x], [_total], color=chart.get('color') or None)
        plt.ylabel(f'Sum of {x}')
    else:
        df[x].dropna().plot(kind='box')
        plt.xlabel(x)
elif chart_type in {'bar', 'word_frequency'} and chart.get('labels'):
    labels = chart.get('labels', [])
    values = chart.get('values', [])
    _cs = chart.get('colors')
    plt.barh(labels[::-1], values[::-1], color=(_cs[::-1] if _cs else chart.get('color')) or None)
    plt.xlabel((chart.get('y_label') or chart.get('y')) if chart_type == 'bar' else 'Count')
elif chart_type == 'stacked_bar' and chart.get('series'):
    import pandas as _pd
    _labels = chart.get('labels', [])
    _stacked_df = _pd.DataFrame({s['label']: s['values'] for s in chart['series']}, index=_labels)
    _stacked_df.plot(kind='bar', stacked=True, ax=plt.gca())
    plt.xlabel(x)
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title=chart.get('group'), bbox_to_anchor=(1.02, 1), loc='upper left')
elif chart_type == 'wordcloud' and chart.get('words'):
    words = chart.get('words', [])
    labels = [w.get('word') for w in words][:25][::-1]
    values = [w.get('count') for w in words][:25][::-1]
    plt.barh(labels, values, color=chart.get('colors') or chart.get('color') or None)
    plt.xlabel('Count')
elif chart_type == 'choropleth' and chart.get('rows'):
    # Renders as an actual interactive map via Plotly Express, which ships
    # its own world/US-state boundaries -- no geopandas or shapefile needed.
    # Falls back to a static ranked bar chart if plotly isn't installed, or
    # if this map used a custom-uploaded .geojson whose shapes Plotly has no
    # built-in knowledge of.
    value_col = chart.get('value_col')
    name_col = chart.get('name_col')
    level = chart.get('level')
    rows = [r for r in chart['rows'] if r.get(value_col) is not None and r.get(name_col)]
    _NAME_TO_ISO3 = json.loads("{\"Zimbabwe\": \"ZWE\", \"Zambia\": \"ZMB\", \"Yemen\": \"YEM\", \"Vietnam\": \"VNM\", \"Venezuela\": \"VEN\", \"Vatican\": \"VAT\", \"Vanuatu\": \"VUT\", \"Uzbekistan\": \"UZB\", \"Uruguay\": \"URY\", \"Federated States of Micronesia\": \"FSM\", \"Marshall Islands\": \"MHL\", \"Northern Mariana Islands\": \"MNP\", \"United States Virgin Islands\": \"VIR\", \"Guam\": \"GUM\", \"American Samoa\": \"ASM\", \"Puerto Rico\": \"PRI\", \"United States of America\": \"USA\", \"South Georgia and the Islands\": \"SGS\", \"British Indian Ocean Territory\": \"IOT\", \"Saint Helena\": \"SHN\", \"Pitcairn Islands\": \"PCN\", \"Anguilla\": \"AIA\", \"Falkland Islands\": \"FLK\", \"Cayman Islands\": \"CYM\", \"Bermuda\": \"BMU\", \"British Virgin Islands\": \"VGB\", \"Turks and Caicos Islands\": \"TCA\", \"Montserrat\": \"MSR\", \"Jersey\": \"JEY\", \"Guernsey\": \"GGY\", \"Isle of Man\": \"IMN\", \"United Kingdom\": \"GBR\", \"United Arab Emirates\": \"ARE\", \"Ukraine\": \"UKR\", \"Uganda\": \"UGA\", \"Turkmenistan\": \"TKM\", \"Turkey\": \"TUR\", \"Tunisia\": \"TUN\", \"Trinidad and Tobago\": \"TTO\", \"Tonga\": \"TON\", \"Togo\": \"TGO\", \"East Timor\": \"TLS\", \"Thailand\": \"THA\", \"United Republic of Tanzania\": \"TZA\", \"Tajikistan\": \"TJK\", \"Taiwan\": \"TWN\", \"Syria\": \"SYR\", \"Switzerland\": \"CHE\", \"Sweden\": \"SWE\", \"Swaziland\": \"SWZ\", \"Suriname\": \"SUR\", \"South Sudan\": \"SSD\", \"Sudan\": \"SDN\", \"Sri Lanka\": \"LKA\", \"Spain\": \"ESP\", \"South Korea\": \"KOR\", \"South Africa\": \"ZAF\", \"Somalia\": \"SOM\", \"Somaliland\": \"SOL\", \"Solomon Islands\": \"SLB\", \"Slovakia\": \"SVK\", \"Slovenia\": \"SVN\", \"Singapore\": \"SGP\", \"Sierra Leone\": \"SLE\", \"Seychelles\": \"SYC\", \"Republic of Serbia\": \"SRB\", \"Senegal\": \"SEN\", \"Saudi Arabia\": \"SAU\", \"S\u00e3o Tom\u00e9 and Principe\": \"STP\", \"San Marino\": \"SMR\", \"Samoa\": \"WSM\", \"Saint Vincent and the Grenadines\": \"VCT\", \"Saint Lucia\": \"LCA\", \"Saint Kitts and Nevis\": \"KNA\", \"Rwanda\": \"RWA\", \"Russia\": \"RUS\", \"Romania\": \"ROU\", \"Qatar\": \"QAT\", \"Portugal\": \"PRT\", \"Poland\": \"POL\", \"Philippines\": \"PHL\", \"Peru\": \"PER\", \"Paraguay\": \"PRY\", \"Papua New Guinea\": \"PNG\", \"Panama\": \"PAN\", \"Palau\": \"PLW\", \"Pakistan\": \"PAK\", \"Oman\": \"OMN\", \"Norway\": \"NOR\", \"North Korea\": \"PRK\", \"Nigeria\": \"NGA\", \"Niger\": \"NER\", \"Nicaragua\": \"NIC\", \"New Zealand\": \"NZL\", \"Niue\": \"NIU\", \"Cook Islands\": \"COK\", \"Netherlands\": \"NLD\", \"Aruba\": \"ABW\", \"Cura\u00e7ao\": \"CUW\", \"Nepal\": \"NPL\", \"Nauru\": \"NRU\", \"Namibia\": \"NAM\", \"Mozambique\": \"MOZ\", \"Morocco\": \"MAR\", \"Western Sahara\": \"ESH\", \"Montenegro\": \"MNE\", \"Mongolia\": \"MNG\", \"Moldova\": \"MDA\", \"Monaco\": \"MCO\", \"Mexico\": \"MEX\", \"Mauritius\": \"MUS\", \"Mauritania\": \"MRT\", \"Malta\": \"MLT\", \"Mali\": \"MLI\", \"Maldives\": \"MDV\", \"Malaysia\": \"MYS\", \"Malawi\": \"MWI\", \"Madagascar\": \"MDG\", \"Macedonia\": \"MKD\", \"Luxembourg\": \"LUX\", \"Lithuania\": \"LTU\", \"Liechtenstein\": \"LIE\", \"Libya\": \"LBY\", \"Liberia\": \"LBR\", \"Lesotho\": \"LSO\", \"Lebanon\": \"LBN\", \"Latvia\": \"LVA\", \"Laos\": \"LAO\", \"Kyrgyzstan\": \"KGZ\", \"Kuwait\": \"KWT\", \"Kosovo\": \"KOS\", \"Kiribati\": \"KIR\", \"Kenya\": \"KEN\", \"Kazakhstan\": \"KAZ\", \"Jordan\": \"JOR\", \"Japan\": \"JPN\", \"Jamaica\": \"JAM\", \"Italy\": \"ITA\", \"Israel\": \"ISR\", \"Palestine\": \"PSE\", \"Ireland\": \"IRL\", \"Iraq\": \"IRQ\", \"Iran\": \"IRN\", \"Indonesia\": \"IDN\", \"India\": \"IND\", \"Iceland\": \"ISL\", \"Hungary\": \"HUN\", \"Honduras\": \"HND\", \"Haiti\": \"HTI\", \"Guyana\": \"GUY\", \"Guinea-Bissau\": \"GNB\", \"Guinea\": \"GIN\", \"Guatemala\": \"GTM\", \"Grenada\": \"GRD\", \"Greece\": \"GRC\", \"Ghana\": \"GHA\", \"Germany\": \"DEU\", \"Georgia\": \"GEO\", \"Gambia\": \"GMB\", \"Gabon\": \"GAB\", \"France\": \"FRA\", \"Saint Pierre and Miquelon\": \"SPM\", \"Wallis and Futuna\": \"WLF\", \"Saint Martin\": \"MAF\", \"Saint Barthelemy\": \"BLM\", \"French Polynesia\": \"PYF\", \"New Caledonia\": \"NCL\", \"French Southern and Antarctic Lands\": \"ATF\", \"Aland\": \"ALA\", \"Finland\": \"FIN\", \"Fiji\": \"FJI\", \"Ethiopia\": \"ETH\", \"Estonia\": \"EST\", \"Eritrea\": \"ERI\", \"Equatorial Guinea\": \"GNQ\", \"El Salvador\": \"SLV\", \"Egypt\": \"EGY\", \"Ecuador\": \"ECU\", \"Dominican Republic\": \"DOM\", \"Dominica\": \"DMA\", \"Djibouti\": \"DJI\", \"Greenland\": \"GRL\", \"Faroe Islands\": \"FRO\", \"Denmark\": \"DNK\", \"Czech Republic\": \"CZE\", \"Northern Cyprus\": \"CYN\", \"Cyprus\": \"CYP\", \"Cuba\": \"CUB\", \"Croatia\": \"HRV\", \"Ivory Coast\": \"CIV\", \"Costa Rica\": \"CRI\", \"Democratic Republic of the Congo\": \"COD\", \"Republic of the Congo\": \"COG\", \"Comoros\": \"COM\", \"Colombia\": \"COL\", \"China\": \"CHN\", \"Macao S.A.R\": \"MAC\", \"Hong Kong S.A.R.\": \"HKG\", \"Chile\": \"CHL\", \"Chad\": \"TCD\", \"Central African Republic\": \"CAF\", \"Cape Verde\": \"CPV\", \"Canada\": \"CAN\", \"Cameroon\": \"CMR\", \"Cambodia\": \"KHM\", \"Myanmar\": \"MMR\", \"Burundi\": \"BDI\", \"Burkina Faso\": \"BFA\", \"Bulgaria\": \"BGR\", \"Brunei\": \"BRN\", \"Brazil\": \"BRA\", \"Botswana\": \"BWA\", \"Bosnia and Herzegovina\": \"BIH\", \"Bolivia\": \"BOL\", \"Bhutan\": \"BTN\", \"Benin\": \"BEN\", \"Belize\": \"BLZ\", \"Belgium\": \"BEL\", \"Belarus\": \"BLR\", \"Barbados\": \"BRB\", \"Bangladesh\": \"BGD\", \"Bahrain\": \"BHR\", \"The Bahamas\": \"BHS\", \"Azerbaijan\": \"AZE\", \"Austria\": \"AUT\", \"Australia\": \"AUS\", \"Indian Ocean Territories\": \"AUS\", \"Heard Island and McDonald Islands\": \"HMD\", \"Norfolk Island\": \"NFK\", \"Ashmore and Cartier Islands\": \"AUS\", \"Armenia\": \"ARM\", \"Argentina\": \"ARG\", \"Antigua and Barbuda\": \"ATG\", \"Angola\": \"AGO\", \"Andorra\": \"AND\", \"Algeria\": \"DZA\", \"Albania\": \"ALB\", \"Afghanistan\": \"AFG\", \"Siachen Glacier\": \"KAS\", \"Antarctica\": \"ATA\", \"Sint Maarten\": \"SXM\", \"Tuvalu\": \"TUV\"}")
    _ISO3_TO_NAME = json.loads("{\"ZWE\": \"zimbabwe\", \"ZMB\": \"zambia\", \"YEM\": \"yemen\", \"VNM\": \"vietnam\", \"VEN\": \"venezuela\", \"VAT\": \"vatican\", \"VUT\": \"vanuatu\", \"UZB\": \"uzbekistan\", \"URY\": \"uruguay\", \"FSM\": \"federated states of micronesia\", \"MHL\": \"marshall islands\", \"MNP\": \"northern mariana islands\", \"VIR\": \"united states virgin islands\", \"GUM\": \"guam\", \"ASM\": \"american samoa\", \"PRI\": \"puerto rico\", \"USA\": \"united states of america\", \"SGS\": \"south georgia and the islands\", \"IOT\": \"british indian ocean territory\", \"SHN\": \"saint helena\", \"PCN\": \"pitcairn islands\", \"AIA\": \"anguilla\", \"FLK\": \"falkland islands\", \"CYM\": \"cayman islands\", \"BMU\": \"bermuda\", \"VGB\": \"british virgin islands\", \"TCA\": \"turks and caicos islands\", \"MSR\": \"montserrat\", \"JEY\": \"jersey\", \"GGY\": \"guernsey\", \"IMN\": \"isle of man\", \"GBR\": \"united kingdom\", \"ARE\": \"united arab emirates\", \"UKR\": \"ukraine\", \"UGA\": \"uganda\", \"TKM\": \"turkmenistan\", \"TUR\": \"turkey\", \"TUN\": \"tunisia\", \"TTO\": \"trinidad and tobago\", \"TON\": \"tonga\", \"TGO\": \"togo\", \"TLS\": \"east timor\", \"THA\": \"thailand\", \"TZA\": \"united republic of tanzania\", \"TJK\": \"tajikistan\", \"TWN\": \"taiwan\", \"SYR\": \"syria\", \"CHE\": \"switzerland\", \"SWE\": \"sweden\", \"SWZ\": \"swaziland\", \"SUR\": \"suriname\", \"SSD\": \"south sudan\", \"SDN\": \"sudan\", \"LKA\": \"sri lanka\", \"ESP\": \"spain\", \"KOR\": \"south korea\", \"ZAF\": \"south africa\", \"SOM\": \"somalia\", \"SOL\": \"somaliland\", \"SLB\": \"solomon islands\", \"SVK\": \"slovakia\", \"SVN\": \"slovenia\", \"SGP\": \"singapore\", \"SLE\": \"sierra leone\", \"SYC\": \"seychelles\", \"SRB\": \"republic of serbia\", \"SEN\": \"senegal\", \"SAU\": \"saudi arabia\", \"STP\": \"s\u00e3o tom\u00e9 and principe\", \"SMR\": \"san marino\", \"WSM\": \"samoa\", \"VCT\": \"saint vincent and the grenadines\", \"LCA\": \"saint lucia\", \"KNA\": \"saint kitts and nevis\", \"RWA\": \"rwanda\", \"RUS\": \"russia\", \"ROU\": \"romania\", \"QAT\": \"qatar\", \"PRT\": \"portugal\", \"POL\": \"poland\", \"PHL\": \"philippines\", \"PER\": \"peru\", \"PRY\": \"paraguay\", \"PNG\": \"papua new guinea\", \"PAN\": \"panama\", \"PLW\": \"palau\", \"PAK\": \"pakistan\", \"OMN\": \"oman\", \"NOR\": \"norway\", \"PRK\": \"north korea\", \"NGA\": \"nigeria\", \"NER\": \"niger\", \"NIC\": \"nicaragua\", \"NZL\": \"new zealand\", \"NIU\": \"niue\", \"COK\": \"cook islands\", \"NLD\": \"netherlands\", \"ABW\": \"aruba\", \"CUW\": \"cura\u00e7ao\", \"NPL\": \"nepal\", \"NRU\": \"nauru\", \"NAM\": \"namibia\", \"MOZ\": \"mozambique\", \"MAR\": \"morocco\", \"ESH\": \"western sahara\", \"MNE\": \"montenegro\", \"MNG\": \"mongolia\", \"MDA\": \"moldova\", \"MCO\": \"monaco\", \"MEX\": \"mexico\", \"MUS\": \"mauritius\", \"MRT\": \"mauritania\", \"MLT\": \"malta\", \"MLI\": \"mali\", \"MDV\": \"maldives\", \"MYS\": \"malaysia\", \"MWI\": \"malawi\", \"MDG\": \"madagascar\", \"MKD\": \"macedonia\", \"LUX\": \"luxembourg\", \"LTU\": \"lithuania\", \"LIE\": \"liechtenstein\", \"LBY\": \"libya\", \"LBR\": \"liberia\", \"LSO\": \"lesotho\", \"LBN\": \"lebanon\", \"LVA\": \"latvia\", \"LAO\": \"laos\", \"KGZ\": \"kyrgyzstan\", \"KWT\": \"kuwait\", \"KOS\": \"kosovo\", \"KIR\": \"kiribati\", \"KEN\": \"kenya\", \"KAZ\": \"kazakhstan\", \"JOR\": \"jordan\", \"JPN\": \"japan\", \"JAM\": \"jamaica\", \"ITA\": \"italy\", \"ISR\": \"israel\", \"PSE\": \"palestine\", \"IRL\": \"ireland\", \"IRQ\": \"iraq\", \"IRN\": \"iran\", \"IDN\": \"indonesia\", \"IND\": \"india\", \"ISL\": \"iceland\", \"HUN\": \"hungary\", \"HND\": \"honduras\", \"HTI\": \"haiti\", \"GUY\": \"guyana\", \"GNB\": \"guinea-bissau\", \"GIN\": \"guinea\", \"GTM\": \"guatemala\", \"GRD\": \"grenada\", \"GRC\": \"greece\", \"GHA\": \"ghana\", \"DEU\": \"germany\", \"GEO\": \"georgia\", \"GMB\": \"gambia\", \"GAB\": \"gabon\", \"FRA\": \"france\", \"SPM\": \"saint pierre and miquelon\", \"WLF\": \"wallis and futuna\", \"MAF\": \"saint martin\", \"BLM\": \"saint barthelemy\", \"PYF\": \"french polynesia\", \"NCL\": \"new caledonia\", \"ATF\": \"french southern and antarctic lands\", \"ALA\": \"aland\", \"FIN\": \"finland\", \"FJI\": \"fiji\", \"ETH\": \"ethiopia\", \"EST\": \"estonia\", \"ERI\": \"eritrea\", \"GNQ\": \"equatorial guinea\", \"SLV\": \"el salvador\", \"EGY\": \"egypt\", \"ECU\": \"ecuador\", \"DOM\": \"dominican republic\", \"DMA\": \"dominica\", \"DJI\": \"djibouti\", \"GRL\": \"greenland\", \"FRO\": \"faroe islands\", \"DNK\": \"denmark\", \"CZE\": \"czech republic\", \"CYN\": \"northern cyprus\", \"CYP\": \"cyprus\", \"CUB\": \"cuba\", \"HRV\": \"croatia\", \"CIV\": \"ivory coast\", \"CRI\": \"costa rica\", \"COD\": \"democratic republic of the congo\", \"COG\": \"republic of the congo\", \"COM\": \"comoros\", \"COL\": \"colombia\", \"CHN\": \"china\", \"MAC\": \"macao s.a.r\", \"HKG\": \"hong kong s.a.r.\", \"CHL\": \"chile\", \"TCD\": \"chad\", \"CAF\": \"central african republic\", \"CPV\": \"cape verde\", \"CAN\": \"canada\", \"CMR\": \"cameroon\", \"KHM\": \"cambodia\", \"MMR\": \"myanmar\", \"BDI\": \"burundi\", \"BFA\": \"burkina faso\", \"BGR\": \"bulgaria\", \"BRN\": \"brunei\", \"BRA\": \"brazil\", \"BWA\": \"botswana\", \"BIH\": \"bosnia and herzegovina\", \"BOL\": \"bolivia\", \"BTN\": \"bhutan\", \"BEN\": \"benin\", \"BLZ\": \"belize\", \"BEL\": \"belgium\", \"BLR\": \"belarus\", \"BRB\": \"barbados\", \"BGD\": \"bangladesh\", \"BHR\": \"bahrain\", \"BHS\": \"the bahamas\", \"AZE\": \"azerbaijan\", \"AUT\": \"austria\", \"AUS\": \"ashmore and cartier islands\", \"HMD\": \"heard island and mcdonald islands\", \"NFK\": \"norfolk island\", \"ARM\": \"armenia\", \"ARG\": \"argentina\", \"ATG\": \"antigua and barbuda\", \"AGO\": \"angola\", \"AND\": \"andorra\", \"DZA\": \"algeria\", \"ALB\": \"albania\", \"AFG\": \"afghanistan\", \"KAS\": \"siachen glacier\", \"ATA\": \"antarctica\", \"SXM\": \"sint maarten\", \"TUV\": \"tuvalu\"}")
    _STATE_TO_ABBR = json.loads("{\"Alabama\": \"AL\", \"Alaska\": \"AK\", \"Arizona\": \"AZ\", \"Arkansas\": \"AR\", \"California\": \"CA\", \"Colorado\": \"CO\", \"Connecticut\": \"CT\", \"Delaware\": \"DE\", \"Florida\": \"FL\", \"Georgia\": \"GA\", \"Hawaii\": \"HI\", \"Idaho\": \"ID\", \"Illinois\": \"IL\", \"Indiana\": \"IN\", \"Iowa\": \"IA\", \"Kansas\": \"KS\", \"Kentucky\": \"KY\", \"Louisiana\": \"LA\", \"Maine\": \"ME\", \"Maryland\": \"MD\", \"Massachusetts\": \"MA\", \"Michigan\": \"MI\", \"Minnesota\": \"MN\", \"Mississippi\": \"MS\", \"Missouri\": \"MO\", \"Montana\": \"MT\", \"Nebraska\": \"NE\", \"Nevada\": \"NV\", \"New Hampshire\": \"NH\", \"New Jersey\": \"NJ\", \"New Mexico\": \"NM\", \"New York\": \"NY\", \"North Carolina\": \"NC\", \"North Dakota\": \"ND\", \"Ohio\": \"OH\", \"Oklahoma\": \"OK\", \"Oregon\": \"OR\", \"Pennsylvania\": \"PA\", \"Rhode Island\": \"RI\", \"South Carolina\": \"SC\", \"South Dakota\": \"SD\", \"Tennessee\": \"TN\", \"Texas\": \"TX\", \"Utah\": \"UT\", \"Vermont\": \"VT\", \"Virginia\": \"VA\", \"Washington\": \"WA\", \"West Virginia\": \"WV\", \"Wisconsin\": \"WI\", \"Wyoming\": \"WY\", \"District of Columbia\": \"DC\"}")
    try:
        import plotly.express as px
        if level == 'world':
            # Detect whether the column holds ISO-3 codes (e.g. 'VEN', 'USA')
            # or full/partial country names -- handle both transparently.
            sample_vals = [str(r.get(name_col, '')).strip().upper() for r in rows[:10]]
            col_has_iso3 = sum(1 for v in sample_vals if v in _ISO3_TO_NAME) >= len(sample_vals) // 2
            for r in rows:
                raw = str(r.get(name_col, '')).strip()
                if col_has_iso3:
                    # Column already has ISO-3 codes -- use directly for Plotly;
                    # also set a display-friendly full lowercase name for hover.
                    iso3 = raw.upper()
                    r['_loc'] = iso3 if iso3 in _ISO3_TO_NAME else None
                    r['_display_name'] = _ISO3_TO_NAME.get(iso3, raw.lower())
                else:
                    # Column has country names -- look up the ISO-3 code.
                    r['_loc'] = _NAME_TO_ISO3.get(raw)
                    r['_display_name'] = raw.lower()
            hover_col = '_display_name'
            locationmode, scope = 'ISO-3', 'world'
        elif level == 'us_states':
            for r in rows:
                raw = str(r.get(name_col)).strip()
                r['_loc'] = _STATE_TO_ABBR.get(raw, raw.upper() if len(raw) == 2 else None)
                r['_display_name'] = raw.lower()
            hover_col = '_display_name'
            locationmode, scope = 'USA-states', 'usa'
        else:
            raise ValueError('this map used a custom .geojson -- Plotly has no built-in boundaries for it')
        mapped_rows = [r for r in rows if r.get('_loc')]
        if not mapped_rows:
            raise ValueError("none of the region names matched Plotly's built-in boundaries")
        fig = px.choropleth(
            mapped_rows, locations='_loc', locationmode=locationmode, color=value_col,
            hover_name=hover_col, color_continuous_scale='Viridis', scope=scope,
            title=chart.get('title') or 'AutoDS chart 5',
        )
        skipped = len(rows) - len(mapped_rows)
        if skipped:
            print(f'{skipped} region(s) had no matching Plotly boundary and were left blank on the map.')
        fig.show()
        _plotted = True
    except ImportError:
        print('plotly is not installed (pip install plotly) -- showing a ranked bar chart instead.')
    except Exception as _map_err:
        print(f'Could not build an interactive map ({_map_err}) -- showing a ranked bar chart instead.')
    if not _plotted:
        rows.sort(key=lambda r: r[value_col], reverse=True)
        labels = [str(r.get(name_col, '?')) for r in rows][:20][::-1]
        values = [r[value_col] for r in rows][:20][::-1]
        plt.barh(labels, values, color=chart.get('colors') or chart.get('color') or None)
        plt.xlabel(value_col)
elif x in df.columns:
    _vc = df[x].value_counts(dropna=True).head(15).sort_values()
    _label_colors = dict(zip(chart.get('labels') or [], chart.get('colors') or []))
    _bar_colors = [_label_colors.get(str(lbl), chart.get('color')) for lbl in _vc.index] if _label_colors else chart.get('color')
    _vc.plot(kind='barh', color=_bar_colors or None)
    plt.xlabel('Count')
else:
    print('The recorded chart columns are not present in the exported data.')

if not _plotted:
    plt.title(chart.get('title') or 'AutoDS chart 5')
    plt.tight_layout()
    plt.show()


**Your notes:**

from the chart, in females(F) Macine Learing is the most enrolled course with 27 out of the 52 students enrolled while in Males(M) it is the least enrolled course with only 2 entries.
Data science is the one prefered by most Males with 27 male students enrolled. Web Develpoment is also preffered among male students with 22 enrolled male. 
cyber security is the least enrolled course generally

## Model Training Done

**Why the numbers below may not match AutoDS exactly:** AutoDS tries a larger roster of models than this notebook can (plus, for some winners, extra hyperparameter tuning and, for some datasets, different feature encoding -- target encoding, PCA, or text features -- instead of the simple one-hot + scaling used here). The leaderboard printed for each run below is AutoDS's actual, authoritative result. `train_replay` below re-trains using the SAME model family AutoDS recorded as the winner whenever that's a plain scikit-learn model (Logistic Regression, Random Forest, Extra Trees, Gradient Boosting, Histogram Gradient Boosting, K-Nearest Neighbors, Ridge Regression); for a winner that needs an extra package (XGBoost / LightGBM / CatBoost) or that's a " (Fast)" variant only used on larger datasets, it falls back to Random Forest and says so in its output -- so the replayed score can still land a bit differently, especially for datasets where the real run depended on that tuning or encoding.

In [ ]:
import json
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    ExtraTreesClassifier, ExtraTreesRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    HistGradientBoostingClassifier, HistGradientBoostingRegressor,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Same default hyperparameters AutoDS itself uses for each of these models
# (see backend/automl.py) -- but NOT any extra RandomizedSearchCV tuning
# AutoDS may have applied on top of them for the actual winner.
def _classification_model(name, class_weight):
    return {
        'Logistic Regression': LogisticRegression(max_iter=1000, class_weight=class_weight),
        'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight=class_weight),
        'Extra Trees': ExtraTreesClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight=class_weight),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=120, max_depth=4),
        'Histogram Gradient Boosting': HistGradientBoostingClassifier(random_state=42, class_weight=class_weight, early_stopping=True, n_iter_no_change=20),
        'K-Nearest Neighbors': KNeighborsClassifier(),
    }.get(name, RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight=class_weight))

def _regression_model(name):
    return {
        'Ridge Regression': Ridge(),
        'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
        'Extra Trees': ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(random_state=42, n_estimators=120, max_depth=4),
        'Histogram Gradient Boosting': HistGradientBoostingRegressor(random_state=42, early_stopping=True, n_iter_no_change=20),
        'K-Nearest Neighbors': KNeighborsRegressor(),
    }.get(name, RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))

_REPLAYABLE_MODEL_NAMES = {
    'Logistic Regression', 'Random Forest', 'Extra Trees', 'Gradient Boosting',
    'Histogram Gradient Boosting', 'K-Nearest Neighbors', 'Ridge Regression',
}

def train_replay(df, target, problem_type, model_name='Random Forest'):
    data = df.dropna(subset=[target]).copy()
    X = data.drop(columns=[target])
    y = data[target]
    numeric_cols = X.select_dtypes(include='number').columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
    ])
    if problem_type == 'classification':
        counts = y.value_counts()
        class_ratio = counts.max() / max(counts.min(), 1)
        class_weight = 'balanced' if class_ratio > 3.0 else None
        model = _classification_model(model_name, class_weight)
        stratify = y if counts.min() >= 2 else None
    else:
        model = _regression_model(model_name)
        stratify = None
    if model_name in _REPLAYABLE_MODEL_NAMES:
        print(f'replaying: {model_name}')
    else:
        print(f'replaying: Random Forest (fallback -- "{model_name}" needs an extra package or only runs on larger datasets)')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=stratify)
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    if problem_type == 'classification':
        print('accuracy:', accuracy_score(y_test, preds))
        print('f1_weighted:', f1_score(y_test, preds, average='weighted', zero_division=0))
    else:
        print('mae:', mean_absolute_error(y_test, preds))
        print('r2:', r2_score(y_test, preds))
    return pipe


In [ ]:
run = json.loads("{\"run_id\": \"current\", \"name\": \"Current training: course\", \"created_at\": null, \"target\": \"course\", \"problem_type\": \"classification\", \"best_model_name\": \"Stacking Ensemble\", \"feature_columns\": [\"student_id\", \"age\", \"gender\", \"enrollment_date\", \"total_payment\", \"full_name\"], \"leaderboard\": [{\"model\": \"Stacking Ensemble\", \"metrics\": {\"accuracy\": 0.76, \"f1_weighted\": 0.7481, \"f1_macro\": 0.7171, \"precision_weighted\": 0.7483, \"recall_weighted\": 0.76, \"per_class_recall\": {\"0\": 1.0, \"1\": 0.25, \"2\": 1.0, \"3\": 0.8333, \"4\": 0.6}, \"roc_auc\": 0.9291}, \"primary_score\": 0.7481}, {\"model\": \"XGBoost\", \"metrics\": {\"accuracy\": 0.72, \"f1_weighted\": 0.7067, \"f1_macro\": 0.6788, \"precision_weighted\": 0.7111, \"recall_weighted\": 0.72, \"per_class_recall\": {\"0\": 0.6667, \"1\": 0.25, \"2\": 1.0, \"3\": 0.8333, \"4\": 0.6}, \"roc_auc\": 0.9348}, \"primary_score\": 0.7067, \"cv_score\": 0.7088}, {\"model\": \"Random Forest\", \"metrics\": {\"accuracy\": 0.76, \"f1_weighted\": 0.7349, \"f1_macro\": 0.6695, \"precision_weighted\": 0.8111, \"recall_weighted\": 0.76, \"per_class_recall\": {\"0\": 0.3333, \"1\": 0.25, \"2\": 1.0, \"3\": 0.8333, \"4\": 1.0}, \"roc_auc\": 0.9229}, \"primary_score\": 0.7349, \"cv_score\": 0.6775}, {\"model\": \"LightGBM\", \"metrics\": {\"accuracy\": 0.68, \"f1_weighted\": 0.6268, \"f1_macro\": 0.5235, \"precision_weighted\": 0.5844, \"recall_weighted\": 0.68, \"per_class_recall\": {\"0\": 0.0, \"1\": 0.0, \"2\": 1.0, \"3\": 0.8333, \"4\": 1.0}, \"roc_auc\": 0.9201}, \"primary_score\": 0.6268, \"cv_score\": 0.6681}, {\"model\": \"Gradient Boosting\", \"metrics\": {\"accuracy\": 0.72, \"f1_weighted\": 0.6925, \"f1_macro\": 0.6294, \"precision_weighted\": 0.6889, \"recall_weighted\": 0.72, \"per_class_recall\": {\"0\": 0.6667, \"1\": 0.0, \"2\": 1.0, \"3\": 0.8333, \"4\": 0.8}, \"roc_auc\": 0.9252}, \"primary_score\": 0.6925, \"cv_score\": 0.6675}, {\"model\": \"Histogram Gradient Boosting\", \"metrics\": {\"accuracy\": 0.68, \"f1_weighted\": 0.6408, \"f1_macro\": 0.5802, \"precision_weighted\": 0.6178, \"recall_weighted\": 0.68, \"per_class_recall\": {\"0\": 0.6667, \"1\": 0.0, \"2\": 1.0, \"3\": 0.8333, \"4\": 0.6}, \"roc_auc\": 0.9304}, \"primary_score\": 0.6408, \"cv_score\": 0.6538}, {\"model\": \"Extra Trees\", \"metrics\": {\"accuracy\": 0.72, \"f1_weighted\": 0.6999, \"f1_macro\": 0.6388, \"precision_weighted\": 0.792, \"recall_weighted\": 0.72, \"per_class_recall\": {\"0\": 0.3333, \"1\": 0.25, \"2\": 1.0, \"3\": 0.6667, \"4\": 1.0}, \"roc_auc\": 0.9145}, \"primary_score\": 0.6999, \"cv_score\": 0.6372}, {\"model\": \"CatBoost\", \"metrics\": {\"accuracy\": 0.64, \"f1_weighted\": 0.5862, \"f1_macro\": 0.4871, \"precision_weighted\": 0.5526, \"recall_weighted\": 0.64, \"per_class_recall\": {\"0\": 0.0, \"1\": 0.0, \"2\": 1.0, \"3\": 0.6667, \"4\": 1.0}, \"roc_auc\": 0.9256}, \"primary_score\": 0.5862, \"cv_score\": 0.6052}, {\"model\": \"K-Nearest Neighbors\", \"metrics\": {\"accuracy\": 0.6, \"f1_weighted\": 0.5753, \"f1_macro\": 0.5241, \"precision_weighted\": 0.5683, \"recall_weighted\": 0.6, \"per_class_recall\": {\"0\": 0.3333, \"1\": 0.25, \"2\": 1.0, \"3\": 0.6667, \"4\": 0.4}, \"roc_auc\": 0.8614}, \"primary_score\": 0.5753, \"cv_score\": 0.599}, {\"model\": \"Logistic Regression\", \"metrics\": {\"accuracy\": 0.56, \"f1_weighted\": 0.5502, \"f1_macro\": 0.4422, \"precision_weighted\": 0.545, \"recall_weighted\": 0.56, \"per_class_recall\": {\"0\": 0.0, \"1\": 0.0, \"2\": 1.0, \"3\": 0.8333, \"4\": 0.4}, \"roc_auc\": 0.9002}, \"primary_score\": 0.5502, \"cv_score\": 0.5941}], \"is_current\": true}")

print(run['name'])
print('target:', run['target'])
print('problem_type:', run['problem_type'])
print('best_model_recorded_by_autods:', run['best_model_name'])
leaderboard = pd.DataFrame(run['leaderboard'])
display(leaderboard)

# Runnable replay using the exported final data and the same target --
# the leaderboard above, not this cell's printed score, is AutoDS's
# authoritative result (see the markdown note above this section).
model = train_replay(df, run['target'], run['problem_type'], run['best_model_name'])


## Predictions Done

In [ ]:
import json

saved_predictions = json.loads("[{\"created_at\": \"2026-10-20 10:06\", \"target\": \"course\", \"problem_type\": \"classification\", \"model_name\": \"Stacking Ensemble\", \"source_name\": \"Current training\", \"run_id\": null, \"inputs\": [{\"student_id\": 602, \"age\": 22, \"gender\": \"F\", \"enrollment_date\": \"2022-04-22\", \"total_payment\": 1300, \"full_name\": \"Agnes Wanjiku\"}], \"predictions\": [\"Machine Learning\"], \"narrative\": \"Given student_id=602, age=22, gender=F, enrollment_date=2022-04-22, total_payment=1300, full_name=Agnes Wanjiku, Stacking Ensemble predicts 'course' = Machine Learning.\"}, {\"created_at\": \"2026-10-20 10:06\", \"target\": \"course\", \"problem_type\": \"classification\", \"model_name\": \"Stacking Ensemble\", \"source_name\": \"Current training\", \"run_id\": null, \"inputs\": [{\"student_id\": 709, \"age\": 23, \"gender\": \"M\", \"enrollment_date\": \"2022-04-22\", \"total_payment\": 1300, \"full_name\": \"Jackson Derrick\"}], \"predictions\": [\"Machine Learning\"], \"narrative\": \"Given student_id=709, age=23, gender=M, enrollment_date=2022-04-22, total_payment=1300, full_name=Jackson Derrick, Stacking Ensemble predicts 'course' = Machine Learning.\"}, {\"created_at\": \"2026-10-20 10:07\", \"target\": \"course\", \"problem_type\": \"classification\", \"model_name\": \"Stacking Ensemble\", \"source_name\": \"Current training\", \"run_id\": null, \"inputs\": [{\"student_id\": 709, \"age\": 23, \"gender\": \"M\", \"enrollment_date\": \"2021-04-22\", \"total_payment\": 1300, \"full_name\": \"Jackson Derrick\"}], \"predictions\": [\"Data Analysis\"], \"narrative\": \"Given student_id=709, age=23, gender=M, enrollment_date=2021-04-22, total_payment=1300, full_name=Jackson Derrick, Stacking Ensemble predicts 'course' = Data Analysis.\"}]")

for prediction in saved_predictions:
    print('target:', prediction.get('target'))
    print('model:', prediction.get('model_name'))
    print('inputs:', prediction.get('inputs'))
    print('predictions:', prediction.get('predictions'))
    print()
